In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/S06_reduccion_dim"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesión 6 — Reducción de dimensionalidad: PCA, Análisis Factorial, t-SNE y UMAP

**Curso:** Herramientas para la Ciencia de Datos, Facultad de Negocios, UPC
**Programa:** Administración y Ciencia de Datos para Negocios

> **Cómo se abre este cuaderno.** El curso lo distribuye por **Google Drive**: en la
> carpeta compartida, clic derecho sobre el archivo → *Abrir con* → *Google
> Colaboratory*. Conviene empezar por **Archivo → Guardar una copia en Drive** para
> conservar el trabajo. No se requiere cuenta de GitHub ni instalar nada en el equipo:
> los datos de la sesión viajan dentro del propio cuaderno.
> **Carpeta del curso en Drive (Pregrado):** https://drive.google.com/drive/folders/1-YJxRt0n-UZwQCu03Lls2LGUYz6KMsl2


---

## Objetivos de aprendizaje (Sección 1 del cuaderno)

Al terminar la sesión, el estudiante:

- Calcula e interpreta **componentes principales** y la **varianza explicada**, y decide cuántos componentes retener (scree plot, criterio de Kaiser, umbral de varianza).
- Usa el PCA para **combatir la multicolinealidad** mediante **PCR** y **PLS** (puente con la Sesión 4).
- Distingue el **PCA** del **Análisis Factorial** y extrae **variables latentes** interpretables con rotación varimax.
- Visualiza datos de alta dimensión con **t-SNE** y **UMAP**, reconociendo sus **límites**.
- Decide **qué técnica de reducción** aplicar según el objetivo: **modelado** (PCA/PCR) o **visualización** (t-SNE/UMAP).

## 2. Mapa de la sesión: nueve capítulos en dos clases

La sesión ocupa **dos clases**. Cada capítulo lleva un código —6.1 a 6.9— que es **el mismo** en el sílabo, en la guía del docente, en la guía de laboratorio y en las diapositivas, de modo que se pueda pasar de un material a otro sin traducir numeraciones.

**JUEVES — 145 min de contenido** (bloque A1 de 75, receso de 15, bloque A2 de 70; antes, 20 min de control sobre la Sesión 5)

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **6.1** | ¿Por qué doscientas columnas correlacionadas continúan siendo un problema? | Sin celdas: se abre en clase con las doscientas columnas correlacionadas |
| **6.2** | ¿Qué es una componente principal? | «Teoría guiada» — eigenvalores, PCA, varianza explicada y criterios de retención |
| **6.3** | ¿Cómo se obtienen las componentes? | «Teoría guiada» — eigenvalores, PCA, varianza explicada y criterios de retención |
| **6.4** | ¿Cuántas componentes se retienen? | «Teoría guiada» — eigenvalores, PCA, varianza explicada y criterios de retención |
| **6.5** | ¿Se sostiene con datos reales? La réplica de t-SNE (2008) | «La réplica»: PCA de Hotelling sobre Iris (cifra exacta) y «Visualización con t-SNE y UMAP sobre digits» (mapa + proxy medible) — laboratorio, pasos 0 a 5 |

**VIERNES — 120 min corridos**

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **6.6** | ¿Cuándo se puede confiar en una proyección? | «Supuestos: decisiones y condiciones de validez» |
| **6.7** | ¿Cómo se verifica que el resultado es real? | «Fidelidad de un embedding: trustworthiness» |
| **6.8** | ¿Qué decisión habilita? PCR sobre el conjunto de S04 | «Laboratorio: PCR sobre el conjunto de S04» — laboratorio, paso 6 |
| **6.9** | ¿Qué no se puede afirmar, y qué sigue en S07? | «Cierre» |

> El **control** de esta sesión se resuelve en aula, en la franja de 20 minutos del jueves siguiente, y cubre **los nueve capítulos**, de los dos días.


## Cómo leer este cuaderno

Este cuaderno **no solo se ejecuta: enseña cada paso**. Se usan estos marcadores:

- ❓ **Qué se quiere averiguar** — abre cada resultado importante: la pregunta que ese número contesta, qué decisión depende de ella y **qué significaría cada resultado posible, dicho antes de ver la cifra**. Conviene detenerse ahí y contestar mentalmente antes de ejecutar: un dato solo informa a quien traía una pregunta.
- 🔎 **Qué hace este código** — explicación breve **antes** de cada celda de código.
- 📖 **Cómo se lee esta salida** — lectura de negocio/estadística **después** de cada resultado clave.
- 💡 **Intuición** — una idea o ejemplo que fija el concepto. ⚠️ **Alerta/supuesto** — un riesgo a vigilar.
- 🖐️ **Cálculo manual** — el alumno reconstruye la mecánica y la verifica contra la librería con `assert`.
- ✅ **Verificación desde la base** — se recomputa el resultado clave desde los datos y se cruza con el Excel (`assert`).
- 🧮 **Matemática en el cuerpo** — la derivación en LaTeX donde se aplica.
- 🧱 **Construcción desde cero** — se rearma el flujo sin los ayudantes de alto nivel y se reproduce el mismo resultado (`assert`).
- 📄 **En el paper** — la procedencia exacta (fuente, sección, valor) del resultado replicado.

**Operativo vs. benchmark.** El valor **operativo** es el que produce este venv al ejecutar; el **benchmark** es el valor publicado (paper/galería de sklearn) y se etiqueta como tal. Cuando difieren por versión de librería o `ddof`, prevalece el operativo y el benchmark queda como referencia con su tolerancia.

**Convención del Excel.** Los **resultados** se vuelcan a `resultados/S06_resultados.xlsx` (openpyxl) en la Sección 6 y las **figuras de resultados se generan leyendo ese Excel**. Los diagnósticos de la Sección 8 (**Supuestos**) **no** escriben en el Excel. La validación numérica con tolerancias vive en el material de referencia de la sesión, que **recomputa desde la base** y cruza recomputado ≈ paper ≈ Excel.

> Marca de sección: en el material se escribe siempre «Sección N» en palabras, nunca el signo tipográfico de sección.

## Preparación del entorno

🔎 **Qué hace este código.** Instala en Colab las **nueve** librerías de la sesión con la versión certificada en la matriz de versiones certificada del curso (`numpy`, `pandas`, `scipy`, `matplotlib`, `scikit-learn`, `statsmodels`, `factor_analyzer`, `openTSNE`, `umap-learn`) **paquete a paquete**: si un tag de versión no resuelve, ese paquete —y solo ese— se reinstala sin fijar, y el resto continúa. Al final imprime la tabla **certificada → instalada** para que cualquier desajuste quede a la vista antes de ejecutar nada. En local se salta la instalación porque ya están en el venv.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys

if "google.colab" in sys.modules:
    %pip install -q factor_analyzer openTSNE umap-learn

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el metodo y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "matplotlib": "3.11.1",
    "numpy": "2.5.1",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "scipy": "1.16.3",
    "statsmodels": "0.14.6",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


🔎 **Qué hace este código.** Importa `numpy`/`pandas`/`matplotlib` y las clases de `sklearn`, `factor_analyzer`, `openTSNE` y `umap`; fija la semilla 42 y la paleta UPC, y define `mostrar()` para guardar cada figura como PNG y mostrarla.

In [ ]:
# Configuración e imports
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")            # backend headless: las figuras se guardan como PNG
import matplotlib.pyplot as plt
from IPython.display import Image, display

from sklearn.datasets import load_iris, load_wine, load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.manifold import trustworthiness
from factor_analyzer import FactorAnalyzer
from openTSNE import TSNE as TSNE_open
import umap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paleta del curso
UPC_ROJO, UPC_TINTA, UPC_GRIS = "#C8102E", "#1F2A44", "#8A8D8F"
PALETA = [UPC_ROJO, UPC_TINTA, "#E4879C", "#5B6472", "#A31621", "#B0B3B5"]
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})

def mostrar(fig, ruta):
    "Guarda la figura como PNG y la muestra en el notebook (backend Agg)."
    try:
        if not fig.get_constrained_layout():
            fig.tight_layout()
    except Exception:
        pass
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close(fig)
    display(Image(str(ruta)))

print("Librerias cargadas. Todo listo para reducir dimensiones.")

🔎 **Qué hace este código.** Localiza la carpeta de la sesión (funciona en local y en Colab), fija las rutas de `resultados/` y `figuras/`, y define `cargar_communities()` para el dataset de negocio de la Sesión 5.

In [ ]:
# Localización de carpetas de la sesión (funciona en local y en Colab)
def localizar_sesion():
    for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        cand = base / "Sesiones" / "S06_reduccion_dim"
        if cand.exists():
            return cand
        if base.name.startswith("S06"):
            return base
    return None

SESION = localizar_sesion()
if SESION is None:
    SESION = Path("/content/S06"); (SESION / "data").mkdir(parents=True, exist_ok=True)
    print("Modo Colab: carpeta de trabajo en", SESION)
else:
    print("Sesion localizada en:", SESION)

RESULTADOS = SESION / "resultados"; RESULTADOS.mkdir(exist_ok=True)
FIGURAS = SESION / "figuras"; FIGURAS.mkdir(exist_ok=True)
XLSX = RESULTADOS / "S06_resultados.xlsx"

def cargar_communities():
    "Communities and Crime (UCI id 183, Redmond 2009): copia local de la Sesion 5 o descarga robusta de UCI."
    # 1) Ruta PREFERIDA y sin red: reutilizar la copia local de la Sesion 5 si existe.
    for base in [SESION.parent, Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        for p in [base / "S05_regularizacion" / "data" / "communities.csv",
                  base / "Sesiones" / "S05_regularizacion" / "data" / "communities.csv"]:
            if p.exists():
                return pd.read_csv(p, na_values="?")
    # 2) Sin copia local (tipico en Colab): descargar de UCI de forma ROBUSTA.
    #    UCI reorganizo su sitio (2023): la ruta legada .../machine-learning-databases/ suele dar 404.
    #    La PCR es el criterio de MAYOR peso del entregable y depende de este dataset, asi que se
    #    intentan varias vias y, si TODAS fallan, se lanza un mensaje ACCIONABLE (no un 404 criptico).
    #    No se sube a OneDrive: la descarga es en memoria (<2 MB), sin cache en disco.
    # Via A (oficial post-migracion): paquete ucimlrepo por id, si esta disponible.
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=183)
        return pd.concat([ds.data.features, ds.data.targets], axis=1)
    except Exception as e_new:  # noqa: BLE001  (sin paquete o sin red): probar el mirror clasico
        print(f"  aviso: via ucimlrepo no disponible ({type(e_new).__name__}); se prueba el mirror clasico de UCI...")
    # Via B (mirror clasico): communities.data + communities.names.
    base_uci = "https://archive.ics.uci.edu/ml/machine-learning-databases/communities/"
    try:
        nombres = [l.split()[1] for l in
                   pd.read_csv(base_uci + "communities.names", header=None, sep="\t")[0]
                   if str(l).lower().startswith("@attribute")]
        return pd.read_csv(base_uci + "communities.data", header=None, names=nombres, na_values="?")
    except Exception as e_leg:  # noqa: BLE001
        raise RuntimeError(
            "No se pudo cargar Communities and Crime (UCI id 183) ni de copia local ni de UCI. "
            "Solucion: coloca 'communities.csv' en Sesiones/S05_regularizacion/data/ (se reutiliza sin "
            "red), o instala 'ucimlrepo' (pip install ucimlrepo) para descargarlo por id, o revisa la "
            f"conexion a archive.ics.uci.edu. Ultimo error del mirror clasico: {e_leg!r}"
        ) from e_leg

print("Resultados ->", RESULTADOS)
print("Figuras    ->", FIGURAS)

## 6.2 a 6.4 — Teoría guiada: qué es una componente principal, cómo se obtiene y cuántas se retienen (Sección 3 del cuaderno)

Cada concepto se presenta con una **minidemostración ejecutable** y su **lectura de negocio**. Las réplicas formales van en la sección 4 y el laboratorio en la 5.

### Eigenvalores y eigenvectores — capítulo 6.3 (subsección 3.1)

🧮 **Matemática en el cuerpo — la matriz de covarianza y el problema de autovalores.**

Sea $X$ la matriz de datos ($n$ observaciones × $p$ variables) y $X_c$ su versión **centrada** (a cada columna se le resta su media). La **matriz de covarianza** es

$$ S \;=\; \frac{1}{\,n-1\,}\,X_c^{\top} X_c \qquad (S\ \text{es } p\times p,\ \text{simétrica}). $$

El PCA resuelve el **problema de autovalores** de $S$:

$$ S\,v_i \;=\; \lambda_i\, v_i, \qquad i=1,\dots,p, $$

donde los **autovectores** $v_1,\dots,v_p$ (ortonormales) son las **direcciones de los componentes principales** y cada **autovalor** $\lambda_i \ge 0$ es la **varianza** que capta su componente. Se ordenan de mayor a menor: $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_p$. Como $S$ es simétrica, `numpy.linalg.eigh` devuelve autovalores reales y autovectores ortogonales.

🔎 **Qué hace este código.** Calcula los autovalores y autovectores de una matriz de covarianza 2×2 y comprueba la identidad $Av=\lambda v$ y la ortogonalidad de los dos ejes.

In [ ]:
# Mini-demo: eigenvectores de una matriz de covarianza cumplen A·v = λ·v
cov = np.array([[3.0, 1.2],
                [1.2, 1.0]])
valores, vectores = np.linalg.eigh(cov)          # eigh: matriz simétrica
orden = np.argsort(valores)[::-1]
valores, vectores = valores[orden], vectores[:, orden]
for i, lam in enumerate(valores):
    v = vectores[:, i]
    print(f"eigenvalor {i+1} = {lam:.3f}  (varianza a lo largo de su eje)  |  A·v = λ·v ? "
          f"{np.allclose(cov @ v, lam * v)}")
print("Los dos eigenvectores son ortogonales (producto punto ≈ 0):",
      round(float(vectores[:, 0] @ vectores[:, 1]), 6))

📖 **Lectura de negocio.** Los eigenvectores de la matriz de covarianza son los **ejes** de los componentes principales y cada eigenvalor dice **cuánta varianza** concentra su eje. "Eigenvalor grande = dirección que concentra mucha variación" es la intuición que justifica quedarse con 2 o 3 ejes en vez de decenas de variables.

### PCA: varianza explicada, scree, Kaiser y biplot — capítulo 6.4 (subsección 3.2)

🧮 **Matemática en el cuerpo — varianza explicada y proyección.**

La **varianza explicada** por el componente $i$ es su autovalor sobre la traza (la suma de todos):

$$ \mathrm{VE}_i \;=\; \frac{\lambda_i}{\sum_{j=1}^{p}\lambda_j}, \qquad \sum_{i=1}^{p}\mathrm{VE}_i = 1. $$

Los datos se **proyectan** sobre los primeros $k$ componentes formando $V_k=[\,v_1\ \cdots\ v_k\,]$ y calculando los *scores*

$$ Z \;=\; X_c\,V_k \qquad (Z\ \text{es } n\times k). $$

**Covarianza vs. correlación.** Trabajar sobre la matriz de **correlación** $R = D^{-1/2}\,S\,D^{-1/2}$, con $D=\operatorname{diag}(S)$, equivale a **estandarizar** las variables (z-score) antes del PCA. Entonces $\sum_i \lambda_i = \operatorname{traza}(R) = p$, y por eso el **criterio de Kaiser** retiene los componentes con $\lambda_i > 1$ (los que explican más que una variable estandarizada). Pasar de $S$ a $R$ **cambia** los autovalores y la varianza explicada: es la **decisión central** de la sesión (ver la guía de supuestos de la sesión, Parte 1.1).

🔎 **Qué hace este código.** Ajusta un PCA a una nube 2D correlacionada y dibuja las flechas de los dos componentes, cada una escalada por su varianza.

In [ ]:
# Mini-demo PCA: una nube 2D correlacionada se resume en su dirección de máxima varianza
rng = np.random.default_rng(RANDOM_STATE)
base = rng.normal(size=(300, 2)) @ np.array([[2.4, 1.6], [0.0, 0.6]])
pca_demo = PCA().fit(base)
print("Varianza explicada por componente:", np.round(pca_demo.explained_variance_ratio_, 3))

fig, ax = plt.subplots(figsize=(5.4, 5.0))
ax.scatter(base[:, 0], base[:, 1], s=12, color=UPC_GRIS, alpha=0.6)
centro = base.mean(axis=0)
for i, (lam, comp) in enumerate(zip(pca_demo.explained_variance_, pca_demo.components_)):
    flecha = comp * np.sqrt(lam) * 2.2
    ax.annotate("", xy=centro + flecha, xytext=centro,
                arrowprops=dict(color=PALETA[i], width=2.4, headwidth=11))
    ax.text(*(centro + flecha * 1.10),
            f"PC{i+1}\n{pca_demo.explained_variance_ratio_[i]*100:.0f}%",
            color=PALETA[i], fontweight="bold", ha="center")
ax.set_title("PCA: PC1 apunta a la dirección de máxima varianza")
ax.set_xlabel("variable 1"); ax.set_ylabel("variable 2"); ax.axis("equal")
mostrar(fig, FIGURAS / "S06_demo_pca_direcciones.png")

📖 **Lectura de negocio.** El PCA recombina variables correlacionadas en ejes ordenados por varianza. Sobre estos ejes se decide **cuántos retener** con tres criterios que se cruzan: el **scree plot** (dónde la curva se aplana), el **criterio de Kaiser** (eigenvalor > 1 sobre datos estandarizados) y el **umbral de varianza acumulada** (p. ej. ≥90 %). El **biplot** convierte un eje abstracto en una historia ("PC1 = tamaño de la flor"). Los tres se aplican sobre **datos reales**, pero no todos en el mismo sitio: el **criterio de Kaiser** y el **umbral de varianza acumulada** salen en la **sección 4** (celdas 41, 54 y 56), mientras que el **scree plot** es una *figura de resultados* y, por la convención Excel del curso, se traza en la **sección 6 leyendo el Excel** (celda 79). El **biplot** aparece en las dos: Iris en la sección 4 (celda 51) y **Wine en la sección 6** (celda 85, leyendo la hoja `cargas_wine`).

### PCR y PLS: recombinar para vencer la multicolinealidad — capítulo 6.8 (subsección 3.3)

🔎 **Qué hace este código.** Con dos predictores casi idénticos, compara el OLS (coeficientes inestables) con una PCR de 1 componente (predicción equivalente).

In [ ]:
# Mini-demo PCR: con dos predictores casi idénticos, el OLS reparte los coeficientes de forma
# inestable; la PCR los recombina en un componente ortogonal y predice con igual calidad.
rng = np.random.default_rng(RANDOM_STATE)
x1 = rng.normal(size=400)
x2 = x1 + rng.normal(scale=0.01, size=400)         # colineal con x1 (corr ≈ 1)
Xd = np.column_stack([x1, x2])
yd = 3.0 * x1 + rng.normal(scale=0.5, size=400)
coef_ols = LinearRegression().fit(Xd, yd).coef_
r2_ols_demo = r2_score(yd, LinearRegression().fit(Xd, yd).predict(Xd))
pcr_demo = Pipeline([("s", StandardScaler()), ("p", PCA(n_components=1)),
                     ("m", LinearRegression())]).fit(Xd, yd)
r2_pcr_demo = r2_score(yd, pcr_demo.predict(Xd))
print(f"Correlación entre x1 y x2: {np.corrcoef(x1, x2)[0, 1]:.4f}")
print(f"Coeficientes OLS (se reparten de forma inestable): {np.round(coef_ols, 2)}")
print(f"R² OLS (2 predictores) = {r2_ols_demo:.3f}  |  R² PCR (1 componente) = {r2_pcr_demo:.3f}")

📖 **Lectura de negocio.** Cuando dos indicadores miden casi lo mismo, el OLS reparte su peso de forma errática (coeficientes grandes de signos opuestos). La **PCR** los funde en un componente ortogonal: **elimina la multicolinealidad** (segunda vía, distinta de la regularización de la Sesión 5) sin sacrificar predicción. **PLS** hace lo mismo pero orienta los componentes hacia la respuesta *Y*, por lo que suele necesitar **menos componentes**.

### Análisis Factorial: variables latentes y rotación varimax — capítulo 6.2 (subsección 3.4)

🔎 **Qué hace este código.** Ajusta un Análisis Factorial de 3 factores con rotación varimax sobre Wine estandarizado e imprime las variables que más cargan en cada factor.

In [ ]:
# Mini-demo Análisis Factorial: 3 factores con rotación varimax sobre Wine estandarizado
wine_demo = load_wine()
Xw_std = StandardScaler().fit_transform(wine_demo.data)
fa_demo = FactorAnalyzer(n_factors=3, rotation="varimax").fit(Xw_std)
cargas = pd.DataFrame(fa_demo.loadings_, index=wine_demo.feature_names,
                      columns=["Factor 1", "Factor 2", "Factor 3"])
print("Variables que MÁS cargan en cada factor (tras varimax, |carga| alta):")
for f in cargas.columns:
    top = cargas[f].abs().sort_values(ascending=False).head(3).index.tolist()
    print(f"  {f}: {top}")

📖 **Lectura de negocio.** El Análisis Factorial no describe la varianza total (como el PCA) sino que **modela variables latentes** —constructos no observables ("cuerpo del vino", "perfil de color")— detrás de las correlaciones. La **rotación varimax** empuja cada variable a cargar fuerte en **un solo** factor, de modo que cada factor se puede **nombrar**. Regla: para **comprimir/quitar multicolinealidad/visualizar** se usa PCA; para **descubrir constructos interpretables**, Análisis Factorial.

### t-SNE: la perplejidad cambia el mapa — capítulo 6.6 (subsección 3.5)

🔎 **Qué hace este código.** Genera dos mapas t-SNE del mismo subconjunto de dígitos con perplejidad 5 y 50, para ver cómo cambia el resultado con ese hiperparámetro.

In [ ]:
# Mini-demo t-SNE: distintas PERPLEJIDADES producen mapas distintos del mismo dato
dig = load_digits()
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(dig.target), size=700, replace=False)
Xsub, ysub = dig.data[idx], dig.target[idx]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True)
for ax, perp in zip(axes, [5, 50]):
    emb = np.asarray(TSNE_open(n_components=2, perplexity=perp, initialization="pca",
                               random_state=RANDOM_STATE, n_jobs=-1).fit(Xsub))
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=ysub, cmap="tab10", s=10, alpha=0.85)
    ax.set_title(f"t-SNE, perplejidad = {perp}")
    ax.set_xlabel("dimensión 1"); ax.set_ylabel("dimensión 2")
fig.colorbar(sc, ax=axes, label="dígito", ticks=range(10))
mostrar(fig, FIGURAS / "S06_demo_tsne_perplejidad.png")

📖 **Lectura de negocio.** La **perplejidad** es el número efectivo de vecinos que t-SNE considera (rango típico 5–50). Con perplejidad baja resalta micro-estructura (puede fragmentar); con alta, estructura más global. **El mismo dato produce mapas distintos** según el valor: nunca se lee un único mapa como verdad absoluta. ⚠️ El mapa cambia **además** con la **semilla** (`random_state`), con el arranque (`initialization`) y —en openTSNE— con `n_jobs` (la reducción en paralelo **no** es reproducible bit-a-bit según el número de núcleos): la deriva **no** se debe solo a la versión de la librería o a la perplejidad. Por eso la reproducibilidad se controla con semilla y se valida con **tolerancias**, no con igualdad exacta.

### UMAP: n_neighbors y min_dist; qué preservan y qué no — capítulo 6.6 (subsección 3.6)

🔎 **Qué hace este código.** Genera dos mapas UMAP variando `n_neighbors` (5 y 50), el hiperparámetro que balancea estructura local y global.

In [ ]:
# Mini-demo UMAP: n_neighbors regula cuánta estructura LOCAL vs GLOBAL se resalta
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True)
for ax, nn in zip(axes, [5, 50]):
    emb = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=0.1,
                    random_state=RANDOM_STATE).fit_transform(Xsub)
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=ysub, cmap="tab10", s=10, alpha=0.85)
    ax.set_title(f"UMAP, n_neighbors = {nn}")
    ax.set_xlabel("dimensión 1"); ax.set_ylabel("dimensión 2")
fig.colorbar(sc, ax=axes, label="dígito", ticks=range(10))
mostrar(fig, FIGURAS / "S06_demo_umap_vecinos.png")

📖 **Lectura de negocio.** En UMAP, **`n_neighbors`** cumple el papel de la perplejidad (local vs. global) y **`min_dist`** regula cuán apretados quedan los puntos (bajo = agrupaciones densas y separadas; alto = reparto uniforme). t-SNE y UMAP **preservan la estructura local** (vecinos cercanos siguen cercanos) pero **NO** son fiables para: la **distancia entre agrupaciones**, el **tamaño** de una agrupación, la **densidad** interna ni un **eje interpretable**; además el mapa **cambia con la semilla**. Son generadores de hipótesis visuales, no conclusiones.

## 6.5 — ¿Se sostiene con datos reales? La réplica: el PCA de Hotelling (1933) sobre Iris y el mapa de dígitos de t-SNE (2008) (Sección 4 del cuaderno)

> **Por qué la réplica tiene dos mitades.** El capítulo replica dos trabajos que producen resultados de naturaleza distinta. De Hotelling se reproduce una **cifra exacta y auditable** —la proporción de varianza explicada por cada componente—, que se contrasta contra el valor canónico. De van der Maaten y Hinton (2008) el resultado publicado es un **mapa**, una figura: no hay número que comparar. Por eso su réplica se cierra con un **proxy medible** —la exactitud de un kNN(5) sobre el embedding 2D y la *trustworthiness*—, que sí se verifica contra tolerancia. El mapa de dígitos se reproduce sobre `digits` (1 797 imágenes de 8×8) y no sobre MNIST-784 (70 000 × 784), demasiado pesado para el aula: la separación de los diez dígitos se obtiene igual. La justificación completa está en la investigación de la sesión, sección «digits y no MNIST-784».

**Paso 0 — Contexto.** Karl Pearson (1901) planteó geométricamente el ajuste de rectas y planos a una nube de puntos; **Hotelling (1933)** lo formalizó como una **eigen-descomposición** de la matriz de covarianza en componentes no correlacionados que capturan sucesivamente la máxima varianza. El **Iris** de Fisher (1936) —150 flores, 4 medidas en cm— se volvió el ejemplo canónico. Se reproduce el valor clásico: **PC1 ≈ 92.46 %** y **PC2 ≈ 5.31 %** (convención de covarianza).

> **Asserts.** En la **réplica** se muestra el valor y se compara contra el target **sin `assert`** (sklearn no reproduce exactamente los datos de R). La verificación numérica con `assert` vive en los bloques **🖐️ (Sección 4.1)**, **✅ (Sección 6.1)** y **🧱 (Sección 7)** —contra la propia base— y en el material de referencia de la sesión con tolerancias.

📄 **En el paper.** El valor que se reproduce es el **PC1 ≈ 92.46 %** (y PC2 ≈ 5.31 %) del PCA de Iris por **covarianza**.

- **Hotelling, H. (1933).** *Analysis of a Complex of Statistical Variables into Principal Components.* Journal of Educational Psychology **24(6):417-441**, DOI 10.1037/h0071325 — formaliza el PCA como **eigen-descomposición** de la matriz de covarianza en componentes no correlacionados de máxima varianza.
- **Pearson, K. (1901).** *On Lines and Planes of Closest Fit to Systems of Points in Space.* Philosophical Magazine, Ser. 6, **2(11):559-572** — el antecedente geométrico (ajustar rectas y planos a una nube de puntos).
- **Valor canónico (benchmark):** `sklearn` (galería `plot_pca_iris` / `plot_pca_vs_lda`) imprime `explained variance ratio (first two components): [0.92461872 0.05306648]`. Contraste obligatorio: por **correlación** (estandarizado) el PC1 baja a **0.7296** (autovalores de la matriz de correlación de Iris: 2.9185, 0.9140, 0.1468, 0.0207). Fuente: la ficha de la sesión de réplica del paper

### Qué preguntaba Pearson, y por qué usó lo que usó — Sección 0 del paper (subsección 4.0)

**💡 Antes de tocar los datos.** Una réplica sin esta pregunta se vuelve mecánica: se ejecutan celdas y se obtiene un número. Lo que sigue explica **qué buscaba el autor del antecedente** y **por qué eligió cada pieza de su método**, que es de donde proviene el criterio para elegir un método propio mañana. *(Desarrollo completo con las citas del original: la ficha de la sesión de réplica del paper, «Sección 0».)*

**El objetivo no era comprimir ni visualizar.** En la primera frase de su artículo Pearson declara: «In many physical, statistical, and biological investigations it is desirable to represent a system of points in plane, three, or higher dimensioned space by the "best-fitting" straight line or plane» (p. 559) —«resulta deseable representar un sistema de puntos en el plano, en tres o en más dimensiones mediante la recta o el plano de "mejor ajuste"»—. Su problema era **geométrico**: ajustar una recta o un plano a una nube de puntos. Comprimir variables y dibujar un mapa en 2D son usos posteriores de la misma solución.

**El defecto que quiso reparar** (p. 559). Los manuales de mínimos cuadrados trataban una variable como respuesta y las demás como conocidas, así que «se obtiene una recta si se trata a una variable como independiente, y otra bien distinta si se trata a otra». Pearson advierte que eso **no es un error** del OLS —dos rectas legítimas responden dos preguntas distintas—, sino un **supuesto** que en física y biología se rompe: «la variable "independiente" está sujeta a tanta desviación o error como la "dependiente"» (pp. 559-560). De ahí su pregunta: **¿cuál es la recta de mejor ajuste cuando ninguna variable merece el rango de respuesta y todas se midieron con error?**

**Por qué la perpendicular** (p. 560). El criterio se declara como decisión, no como verdad: «Of course the term "best fit" is really arbitrary; but a good fit will clearly be obtained if we make the sum of the squares of the perpendiculars […] a minimum». Lo que obtiene es la unicidad: la distancia perpendicular no distingue entre las variables, de modo que la recta ya no depende de cuál se llamó respuesta. Y aporta además el anclaje que se ve en la celda siguiente: la recta de mejor ajuste pasa por el **centroide** (p. 561) — por eso el PCA empieza siempre por centrar.

**Por qué le bastaban medias, desviaciones típicas y correlaciones** (p. 563). «It depends only on a knowledge of the means, standard-deviations, and correlations of the q variables». Dos motivos: esas tres cantidades ya las tenía calculadas desde su memoria de 1896, y la vía elegida «will usually be a process involving much simpler arithmetic» (p. 569), decisivo cuando todo se resuelve de forma manual. Consecuencia moderna: **al PCA le basta la matriz de covarianza o de correlación**, y elegir entre una y otra cambia los porcentajes (la decisión de la Sección 4 de este cuaderno y del bloque 8.1).

**Qué comprobó antes de apoyarse en el método** (p. 563). Si el elipsoide de residuos degenera en esfera, «todo plano que pase por el centroide ajusta igual de bien», y esa esfericidad «implica que se anulan todas las correlaciones y que todas las desviaciones típicas son iguales». Ese es, con otro nombre, el **scree plot plano**: sin correlación y con dispersión pareja no hay dirección privilegiada y no hay nada que reducir.

**⚠️ La diferencia con el paper, dicha de frente.** El artículo de 1901 **no contiene** la proporción de varianza explicada que este cuaderno reproduce: Pearson reporta cosenos directores y un residuo medio cuadrático en las unidades de los datos. La contabilidad adimensional por componente es posterior, de **Hotelling (1933)**. No es una discrepancia de cálculo sino un **cambio de unidad de cuenta**; la conclusión que responde su pregunta —existe una dirección que ajusta la nube casi por completo y el resto es residuo— se reproduce intacta.


**❓ Qué se quiere averiguar.** De cada flor se midieron cuatro variables —largo y ancho de sépalo, largo y ancho de pétalo—. ¿Hacían falta las cuatro, o esas cuatro cifras son una sola magnitud vista desde cuatro ángulos?

- **Qué decide:** si un solo eje resume casi toda la variación, el informe pasa de cuatro columnas a una y la nube vuelve a caber en un gráfico. Si no, cualquier resumen en dos dimensiones esconde algo que después se echará de menos.
- **Antes de mirar el resultado:** con cuatro variables independientes y de igual escala, a cada componente le tocaría un **25 %**; ese es el suelo de «aquí no hay redundancia». Cerca del **100 %**, las cuatro medidas eran casi redundantes. Y hay una referencia externa: este porcentaje es el que Hotelling publicó en 1933, así que la réplica no se aprueba por salir alta, sino por salir **igual a la suya**.

🔎 **Qué hace este código.** Ajusta el PCA de Iris sobre los datos crudos (sklearn centra pero **no** escala → convención de **covarianza**) y lee la varianza explicada de PC1 y PC2.

In [ ]:
# Paso 1: PCA de Iris con la CONVENCIÓN DE COVARIANZA
# (datos crudos; sklearn.PCA centra pero NO escala -> equivale a la matriz de covarianza)
iris = load_iris()
X_iris = iris.data
pca_cov = PCA().fit(X_iris)
iris_cov_evr = pca_cov.explained_variance_ratio_
iris_pc1_cov, iris_pc2_cov = float(iris_cov_evr[0]), float(iris_cov_evr[1])
iris_cov_acum2 = float(iris_cov_evr[:2].sum())
print("Varianza explicada (covarianza):", np.round(iris_cov_evr, 4))
print(f"PC1 = {iris_pc1_cov*100:.2f}%   (target 92.46%)")
print(f"PC2 = {iris_pc2_cov*100:.2f}%    (target 5.31%)")
print(f"PC1 + PC2 = {iris_cov_acum2*100:.2f}%  ->  con 2 componentes se ve Iris en 2D casi sin perder nada")

🔎 **Qué hace este código.** Repite el PCA de Iris tras estandarizar (z-score → convención de **correlación**) y calcula el criterio de Kaiser (autovalores > 1).

In [ ]:
# Paso 2: la MISMA Iris con la CONVENCIÓN DE CORRELACIÓN (StandardScaler -> z-score)
pca_cor = PCA().fit(StandardScaler().fit_transform(X_iris))
iris_cor_evr = pca_cor.explained_variance_ratio_
iris_cor_eig = pca_cor.explained_variance_
iris_pc1_cor, iris_pc2_cor = float(iris_cor_evr[0]), float(iris_cor_evr[1])
iris_cor_acum2 = float(iris_cor_evr[:2].sum())
iris_kaiser = int((iris_cor_eig > 1).sum())
print("Varianza explicada (correlación):", np.round(iris_cor_evr, 4))
print(f"PC1 = {iris_pc1_cor*100:.2f}%   (target 72.96%)  |  PC2 = {iris_pc2_cor*100:.2f}%   (target 22.85%)")
print(f"Eigenvalores (correlación): {np.round(iris_cor_eig, 3)}")
print(f"Criterio de Kaiser (eigenvalor > 1): {iris_kaiser} componente(s)")

📖 **Lectura de la réplica.** La misma Iris da porcentajes distintos según la convención:

| Convención | PC1 | PC2 | PC1+PC2 | Kaiser (λ>1) |
|---|---:|---:|---:|---:|
| **Covarianza** (datos crudos) | **92.46 %** | **5.31 %** | 97.77 % | — |
| **Correlación** (estandarizado) | **72.96 %** | **22.85 %** | 95.81 % | **1** |

Con **covarianza**, la variable de mayor varianza numérica (el largo del pétalo) **domina** el PC1 → 92 %. Con **correlación**, todas las variables parten con el mismo peso y el PC1 baja al 73 %. **Ambas son correctas**; lo obligatorio es **documentar cuál se usa**. Regla práctica: si las unidades difieren (dólares, años, kg), se **debe** estandarizar (correlación). Nótese que **Kaiser sugiere 1** componente mientras el scree y la varianza acumulada sugieren **2**: los criterios pueden discrepar.

### 🖐️ Cálculo manual — abrir el PCA de Iris por dentro — capítulo 6.3 (subsección 4.1)

Las celdas anteriores usaron `sklearn.PCA` como caja negra. Aquí se reconstruye el mismo resultado **paso a paso** —centrar → matriz de covarianza → autovalores/autovectores → proyectar— y se **verifica con `assert`** contra `sklearn` y `statsmodels`. Después se repite por la convención de **correlación** para ver, de forma manual, la **decisión central** de la sesión (0.9246 → 0.7296).

🔎 **Qué hace este código.** Reconstruye el PCA de Iris por covarianza **de forma manual**: centra, calcula $S=X_c^{\top}X_c/(n-1)$, resuelve el problema de autovalores con `eigh`, proyecta, y verifica la varianza explicada contra `sklearn` y `statsmodels`.

In [ ]:
# 🖐️ PCA de Iris "a mano" (convención de COVARIANZA), sin usar sklearn.PCA para el cálculo
from statsmodels.multivariate.pca import PCA as SM_PCA   # tercer testigo, además de numpy y sklearn

Xc = X_iris - X_iris.mean(axis=0)                  # 1) centrar (la varianza se mide respecto a la media)
n = Xc.shape[0]
S = (Xc.T @ Xc) / (n - 1)                          # 2) matriz de covarianza S = Xc^T Xc / (n-1)
lam, Vec = np.linalg.eigh(S)                       # 3) autovalores/autovectores (eigh: S es simetrica)
orden = np.argsort(lam)[::-1]
lam, Vec = lam[orden], Vec[:, orden]               #    ordenar de mayor a menor varianza
evr_mano = lam / lam.sum()                         # 4) varianza explicada VE_i = lambda_i / sum(lambda)
Z_mano = Xc @ Vec[:, :2]                           # 5) proyeccion Z = Xc V (scores de PC1-PC2)

evr_sklearn = pca_cov.explained_variance_ratio_    # de la celda anterior
sm_eig = SM_PCA(X_iris, standardize=False, demean=True, method="eig").eigenvals
evr_sm = np.sort(sm_eig)[::-1] / np.sum(sm_eig)    # statsmodels: razon de autovalores

print(f"a mano   PC1={evr_mano[0]:.4f}  PC2={evr_mano[1]:.4f}   (target 0.9246 / 0.0531)")
print(f"sklearn  PC1={evr_sklearn[0]:.4f}  PC2={evr_sklearn[1]:.4f}")
print(f"statsmod PC1={evr_sm[0]:.4f}  PC2={evr_sm[1]:.4f}")
print(f"varianza de los scores de PC1 = {Z_mano[:, 0].var(ddof=1):.4f}  ==  lambda_1 = {lam[0]:.4f}")

assert abs(evr_mano[0] - 0.9246) <= 0.005                       # reproduce el valor canonico de Hotelling
assert np.allclose(evr_mano[:2], evr_sklearn[:2], atol=1e-6)    # a mano == sklearn
assert np.allclose(evr_mano[:2], evr_sm[:2], atol=1e-4)         # a mano == statsmodels
assert abs(Z_mano[:, 0].var(ddof=1) - lam[0]) <= 1e-6          # la proyeccion tiene varianza lambda_1
print("OK: a mano == sklearn == statsmodels")

📖 **Cómo se lee.** Los tres caminos (manual, `sklearn`, `statsmodels`) dan el **mismo** PC1 = **0.9246** y PC2 = **0.0531**: el PCA es exactamente la eigen-descomposición de la covarianza. Y la varianza de los *scores* de PC1 es igual a su autovalor $\lambda_1$ —la proyección **es** la que maximiza la varianza—. 💡 El signo de un autovector es arbitrario (la librería puede voltearlo); no cambia ni la varianza ni la interpretación.

🔎 **Qué hace este código.** Repite el PCA de forma manual sobre la **matriz de correlación** (equivale a estandarizar) y comprueba que el PC1 cae a 0.7296 y que Kaiser (λ>1) retiene 1 componente.

In [ ]:
# 🖐️ La MISMA Iris por la convencion de CORRELACION: estandarizar == usar la matriz R
R = np.corrcoef(X_iris, rowvar=False)              # matriz de correlacion (equivale al z-score)
lam_R = np.sort(np.linalg.eigvalsh(R))[::-1]       # autovalores de R, de mayor a menor
evr_mano_cor = lam_R / lam_R.sum()                 # suma(lambda) = traza(R) = p = 4
kaiser_mano = int((lam_R > 1).sum())               # criterio de Kaiser: autovalores > 1

print(f"a mano (correlacion)  PC1={evr_mano_cor[0]:.4f}  PC2={evr_mano_cor[1]:.4f}   (target 0.7296 / 0.2285)")
print(f"autovalores (correlacion): {np.round(lam_R, 4)}   suma = {lam_R.sum():.1f} (= n de variables)")
print(f"Kaiser (lambda > 1): {kaiser_mano} componente(s)  |  sklearn dio {iris_kaiser}")

assert abs(evr_mano_cor[0] - 0.7296) <= 0.007                     # el PC1 CAE de 0.9246 a 0.7296
assert np.allclose(evr_mano_cor[:2], iris_cor_evr[:2], atol=1e-6) # a mano == sklearn (celda anterior)
assert kaiser_mano == 1                                           # Kaiser retiene 1 componente
print("OK: la decision covarianza (0.9246) vs correlacion (0.7296) cambia el PC1 ~19 puntos")

📖 **Cómo se lee — la decisión central.** Sobre la **misma** Iris, al estandarizar (matriz de correlación) el PC1 **cae de 0.9246 a 0.7296** y el PC2 sube: ya no domina la variable de mayor escala, todas pesan igual. La suma de autovalores es **4** (= nº de variables), por eso el **criterio de Kaiser** (λ>1) retiene **1** componente. ⚠️ **Ambas convenciones son correctas**; lo obligatorio es **decidir por las unidades y documentar cuál se usa** (ver Sección 8.1 y la guía de supuestos de la sesión, Parte 1.1).

**❓ Qué se quiere averiguar.** El PC1 ya tiene su porcentaje de varianza, pero todavía no tiene nombre. ¿Qué mide ese eje, en palabras que un biólogo —o un gerente— pueda usar?

- **Qué decide:** un componente sin nombre no se reporta ni se acciona. «El PC1 explica el 92,46 %» no es un hallazgo; «el 92,46 % de la variación entre flores es tamaño de pétalo» sí lo es. La distancia entre las dos frases es exactamente este gráfico: la varianza explicada no significa nada de negocio hasta que alguien interpreta el eje.
- **Antes de mirar el resultado:** si las cuatro flechas salieran dispersas y de largos parecidos, el eje sería una mezcla sin lectura y habría que renunciar a nombrarlo. Si **una o dos dominan y se alinean con el eje**, el nombre se deriva de ellas. Conviene además comprometerse con qué número se reportará: para el largo de pétalo, el autovector (**0,8567**), la carga v·√λ (**1,7615 cm**) y la correlación (**0,9979**) son **tres cifras distintas** del mismo hecho, y solo la última está acotada por 1.

🔎 **Qué hace este código.** Proyecta Iris sobre PC1-PC2 y superpone las flechas de las cuatro cargas para poder **nombrar** el eje PC1.

In [ ]:
# Biplot de Iris (PC1-PC2, convención de covarianza): observaciones + flechas de las cargas
Z = pca_cov.transform(X_iris)[:, :2]
cargas_iris = pca_cov.components_[:2].T            # 4 variables x 2 componentes
iris_loadings = cargas_iris.copy()
esc = np.abs(Z).max(axis=0) / np.abs(cargas_iris).max(axis=0) * 0.9
nombres_var = ["largo sépalo", "ancho sépalo", "largo pétalo", "ancho pétalo"]

fig, ax = plt.subplots(figsize=(7.2, 6.0))
for k, especie in enumerate(iris.target_names):
    m = iris.target == k
    ax.scatter(Z[m, 0], Z[m, 1], s=18, alpha=0.7, color=PALETA[k], label=especie)
for j, nombre in enumerate(nombres_var):
    ax.arrow(0, 0, cargas_iris[j, 0]*esc[0], cargas_iris[j, 1]*esc[1],
             color=UPC_TINTA, width=0.012, head_width=0.09, length_includes_head=True)
    ax.text(cargas_iris[j, 0]*esc[0]*1.14, cargas_iris[j, 1]*esc[1]*1.14, nombre,
            color=UPC_TINTA, fontsize=10, ha="center", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75))
ax.axhline(0, color=UPC_GRIS, lw=0.6); ax.axvline(0, color=UPC_GRIS, lw=0.6)
ax.set_xlabel(f"PC1 ({iris_pc1_cov*100:.1f}% de la varianza)")
ax.set_ylabel(f"PC2 ({iris_pc2_cov*100:.1f}% de la varianza)")
ax.set_title("Biplot de Iris: PC1 ≈ tamaño del pétalo / tamaño general de la flor")
ax.legend(title="especie")
mostrar(fig, FIGURAS / "S06_biplot_iris.png")

# --- TRES objetos distintos: autovector unitario, CARGA (v·√λ) y CORRELACIÓN variable-componente ---
# Este PCA es por COVARIANZA (Iris crudo, en cm). En esa convención v·√λ NO es una correlación:
# está en las UNIDADES de la variable y puede pasar de 1. La identidad «carga = correlación» SOLO
# vale con el PCA sobre la matriz de CORRELACIÓN (datos estandarizados, celda 56 con Wine).
# Para volver a la escala de correlación se divide por la desviación típica σ de la variable.
lam_cov = pca_cov.explained_variance_[:2]                     # eigenvalores (varianza por componente)
V_bip = pca_cov.components_[:2].T                             # 4x2 autovectores unitarios
L_bip = V_bip * np.sqrt(lam_cov)                              # 4x2 carga v·√λ (unidades de la variable)
sd_bip = X_iris.std(axis=0, ddof=1)                           # σ de cada variable (cm)
C_bip = L_bip / sd_bip[:, None]                               # 4x2 correlación variable-componente
C_emp = np.corrcoef(X_iris.T, Z.T)[:4, 4:]                    # correlación empírica (control independiente)
tabla_cargas_biplot_iris = [
    [nombres_var[j], float(V_bip[j, 0]), float(L_bip[j, 0]), float(sd_bip[j]), float(C_bip[j, 0]),
     float(C_emp[j, 0]), float(V_bip[j, 1]), float(L_bip[j, 1]), float(C_bip[j, 1]), float(C_emp[j, 1])]
    for j in range(len(nombres_var))
]
jp = nombres_var.index("largo pétalo")                        # variable dominante del PC1
print(f"components_ es el AUTOVECTOR UNITARIO -> norma^2 del PC1 = {(pca_cov.components_[0]**2).sum():.4f} (=1)")
print(f"'largo pétalo' en PC1:  v = {V_bip[jp, 0]:+.4f}  |  carga v·√λ = {L_bip[jp, 0]:+.4f} (cm, escala de "
      f"COVARIANZA)  |  σ = {sd_bip[jp]:.4f} cm  |  correlación = v·√λ/σ = {C_bip[jp, 0]:+.4f}")
print(f"   control independiente: corr(variable, score PC1) = {C_emp[jp, 0]:+.4f}   "
      f"(máx |desvío| en las 4 variables = {np.abs(C_bip - C_emp).max():.6f})")
print("Regla trazable: CARGA = COMPONENTE × √λ. Con COVARIANZA esa carga va en unidades de la variable")
print("(+1.7615 cm, MAYOR QUE 1: no puede ser una correlación); la correlación sale al dividir por σ")
print("(+0.9979). Solo con CORRELACIÓN (datos estandarizados, σ=1) ambas coinciden: Wine flavanoids")
print("v=0.4229 -> carga v·√λ = 0.9201 = correlación. Las 10 columnas de esta tabla se registran en la")
print("hoja `cargas_biplot_iris` del Excel. Ver DEFINICIONES, Sección 6.")

📖 **Lectura del biplot.** Las flechas de **largo y ancho del pétalo** son largas y apuntan casi paralelas al eje PC1 → ese eje se puede nombrar **"tamaño del pétalo / tamaño general de la flor"**. Las especies se ordenan de izquierda a derecha por PC1 (*setosa* con pétalo pequeño a la izquierda; *virginica* con pétalo grande a la derecha). El **signo** de un componente es arbitrario (sklearn puede voltearlo): no se interpreta como bueno/malo.

**Tres objetos distintos, no dos (lo que imprime la celda).** Las flechas se dibujan desde `pca.components_`, el **autovector unitario** (norma 1; para Iris sus tres coordenadas grandes caen en **0,3–0,9** —la del largo de pétalo es 0,8567— y solo el ancho de sépalo queda por debajo, en 0,0845). De ahí salen otros dos números que **no** son intercambiables:

| Objeto | Fórmula | Largo de pétalo (Iris, covarianza) | ¿Acotado por 1? |
|---|---|---:|---|
| Autovector unitario | `pca.components_` = vⱼₖ | **+0,8567** (rango de las cuatro: 0,0845–0,8567) | sí (Σⱼ vⱼₖ² = 1) |
| **Carga** | `vⱼₖ — √λₖ` | **+1,7615 cm** | **no** — va en las unidades de la variable |
| **Correlación** variable-componente | `vⱼₖ — √λₖ / σⱼ` | **+0,9979** | sí, por definición |

⚠️ **La identidad «carga = correlación» solo vale con el PCA sobre la matriz de CORRELACIÓN** (datos estandarizados, donde σⱼ = 1 y el divisor desaparece). Este biplot es por **covarianza**: aquí `v·√λ = +1,7615` es la longitud de la flecha en la escala de la covarianza —en centímetros— y llamarla "correlación" sería imposible, porque ninguna correlación pasa de 1. El factor sobrante es exactamente σ del largo de pétalo (**1,7653 cm**): 1,7615 / 1,7653 = 0,9979, que es la correlación real (y la celda la contrasta contra `corr(variable, score)` calculada aparte, con desvío 0). Con Wine **estandarizado** (celda 56) sí coinciden: `flavanoids` v = 0,4229 → carga 0,9201 = correlación. Quien reproduzca las cargas leyendo `components_` directo obtendrá ~0,42, **no** 0,92: falta el **×√λ**. El orden y el signo de las variables no cambian con ninguna de las tres escalas (√λ y σ son constantes positivas), solo el número que se reporta. Las tres columnas quedan registradas en la hoja **`cargas_biplot_iris`** del Excel; desarrollo en el glosario de la sesión

**❓ Qué se quiere averiguar.** ¿Cuántas dimensiones hacen falta realmente para describir la imagen de un dígito, y qué se acepta perder a cambio de quedarse con menos?

- **Qué decide:** el número de componentes retenidos es el tamaño del índice que se almacena y se consulta. La hoja `costeo_compresion` lo traduce a factura: para un millón de vectores y la tarifa ilustrativa de 0,25 $/GB-mes, 64 dimensiones cuestan 0,0640 $/mes y el índice comprimido baja a 0,0210 $/mes.
- **Antes de mirar el resultado:** si bastaran **2 o 3** componentes, la nube sería casi plana y un mapa en 2D contaría toda la historia. Si hicieran falta **60 de 64**, no habría redundancia y comprimir no compensaría. Cualquier cifra intermedia obliga a nombrar el precio: el objetivo es el 90 % de la varianza, y el **10 % restante es lo que se descarta**. Conviene anticipar también la respuesta para MNIST-784 antes de verla: con 12 veces más píxeles, ¿hacen falta 12 veces más componentes?

🔎 **Qué hace este código.** Cuenta cuántos de los 64 componentes hacen falta para conservar ≥90 % de la varianza en `digits` y —**paso opcional de alta dimensión**— repite exactamente la misma pregunta sobre la muestra **MNIST-784** (`data/mnist_muestra.npz`, 2000×784, ≤1 MB): 12 veces más píxeles, la misma cuenta. Si el archivo no está disponible (p. ej. en Colab sin la carpeta `data/`), el paso se omite sin romper el cuaderno.

In [ ]:
# Paso 3: ¿cuántos componentes para conservar ≥90% de la varianza en digits (64 dimensiones)?
digits = load_digits()
pca_dig = PCA().fit(digits.data)
digits_evr = pca_dig.explained_variance_ratio_
digits_cum = np.cumsum(digits_evr)
digits_n90 = int(np.argmax(digits_cum >= 0.90) + 1)
print(f"digits: {digits_n90} de 64 componentes conservan ≥90% de la varianza   (target 21)")
print(f"PC1 explica {digits_evr[0]*100:.2f}%; con solo 2 componentes apenas {digits_cum[1]*100:.1f}%")

# --- PASO OPCIONAL DE ALTA DIMENSION: la misma pregunta sobre MNIST-784 (28x28 = 784 pixeles) ---
# `digits` es la version 8x8 (64 px). La muestra `data/mnist_muestra.npz` (2000x784, <=1 MB, con
# checksum en descargar_datos.py) permite ver si la conclusion escala a una dimension 12 veces
# mayor SIN traer los ~55 MB del MNIST completo (regla del curso: nada >25 MB en el repositorio).
MNIST_NPZ = SESION / "data" / "mnist_muestra.npz"
if MNIST_NPZ.exists():
    _mn = np.load(MNIST_NPZ)
    X_mnist = _mn["X"].reshape(len(_mn["X"]), -1).astype(float)
    mnist_cum = np.cumsum(PCA().fit(X_mnist).explained_variance_ratio_)
    mnist_n90 = int(np.argmax(mnist_cum >= 0.90) + 1)
    print(f"MNIST-784 (muestra {X_mnist.shape[0]}x{X_mnist.shape[1]}): {mnist_n90} de 784 "
          f"componentes conservan ≥90% -> compresión {X_mnist.shape[1]/mnist_n90:.1f}x "
          f"(digits: {64/digits_n90:.1f}x). Misma lección, dimensión 12 veces mayor.")
else:
    X_mnist, mnist_n90 = None, None
    print("(muestra MNIST-784 no disponible: se omite el paso opcional de alta dimensión)")

**❓ Qué se quiere averiguar.** La respuesta a «cuántos componentes hacen falta», ¿describe el vino o describe las unidades en que alguien decidió medirlo?

- **Qué decide:** la misma tabla de 13 variables puede sostener el informe «un solo indicador resume el vino» o el informe «hacen falta ocho». Solo uno de los dos resiste la pregunta sobre su origen.
- **Antes de mirar el resultado:** `proline` se mide en centenares y `hue` en décimas. Si la escala fuese irrelevante, ambas convenciones darían el mismo número de componentes. Si no lo es, el PCA crudo devolverá un PC1 muy elevado — y ese porcentaje casi perfecto es una **señal de alarma**, no una buena noticia: querrá decir que el componente copió a la variable de números más grandes en vez de encontrar estructura.

🔎 **Qué hace este código.** Contrasta Wine **crudo** vs. **estandarizado**: cuántos componentes cruzan el 90 % en cada convención (la lección de estandarizar).

In [ ]:
# Contraste Wine: la CONVENCIÓN de escalado cambia la conclusión (INTERPRETACION_RESULTADOS Seccion 5)
wine = load_wine()
cum_wine_raw = np.cumsum(PCA().fit(wine.data).explained_variance_ratio_)
cum_wine_std = np.cumsum(PCA().fit(StandardScaler().fit_transform(wine.data)).explained_variance_ratio_)
wine_n90_raw = int(np.argmax(cum_wine_raw >= 0.90) + 1)
wine_n90_std = int(np.argmax(cum_wine_std >= 0.90) + 1)
print(f"Wine CRUDO:         {wine_n90_raw} componente para ≥90% "
      f"(PC1 = {cum_wine_raw[0]*100:.1f}% -> ALARMA: es un artefacto de no estandarizar)")
print(f"Wine ESTANDARIZADO: {wine_n90_std} componentes para ≥90%   (target 8) -> complejidad real del dato")

# --- CARGAS de Wine estandarizado: las 13 variables en PC1 y PC2 -> hoja `cargas_wine` del Excel ---
# Aquí el PCA es sobre la matriz de CORRELACIÓN (datos estandarizados, σ=1), así que la carga v·√λ
# SÍ es la correlación variable-componente y está acotada por 1 — a diferencia del biplot de Iris
# por covarianza (celda 51), donde la misma fórmula da +1.7615 cm y NO es una correlación.
Xw_std = StandardScaler().fit_transform(wine.data)
pca_wine_std = PCA().fit(Xw_std)
lam_w = pca_wine_std.explained_variance_[:2]
V_w = pca_wine_std.components_[:2].T                          # 13x2 autovectores unitarios
L_w = V_w * np.sqrt(lam_w)                                    # 13x2 carga = correlación
tabla_cargas_wine = [[nom, float(V_w[j, 0]), float(L_w[j, 0]), float(V_w[j, 1]), float(L_w[j, 1])]
                     for j, nom in enumerate(wine.feature_names)]
print(f"\nCargas de Wine estandarizado (carga = v·√λ = correlación; λ1 = {lam_w[0]:.4f}, λ2 = {lam_w[1]:.4f}):")
print(f"{'variable':32s}{'v PC1':>9s}{'carga PC1':>11s}{'v PC2':>9s}{'carga PC2':>11s}")
for nom, v1, c1, v2, c2 in sorted(tabla_cargas_wine, key=lambda r: -r[2]):
    print(f"{nom:32s}{v1:>9.4f}{c1:>+11.4f}{v2:>9.4f}{c2:>+11.4f}")
print("PC1 = RIQUEZA FENÓLICA: flavanoids +0.92, total_phenols +0.86, od280 +0.82, proanthocyanins")
print("     +0.68 y hue +0.65, contra nonflavanoid_phenols -0.65 y malic_acid -0.53.")
print("PC2 = CUERPO / POTENCIA FÍSICA: color_intensity +0.84, alcohol +0.77, proline +0.58.")
print("OJO, error típico del Drill 2: `alcohol` NO carga en PC1 (+0.31) sino en PC2 (+0.77), y `hue`")
print("NO carga en PC2 (-0.44) sino en PC1 (+0.65). El eje se nombra por lo que las cargas dicen, no")
print("por lo que la intuición enológica sugiere.")

📖 **Lectura.** En **digits** (64 píxeles en la misma escala 0–16) hacen falta **21** componentes para el 90 %: los datos son genuinamente multidimensionales. En **Wine** (13 variables de escalas dispares), sin estandarizar "1 componente basta" (99.8 %) es un **defecto de preprocesamiento** —`proline`, con valores numéricos muy elevados, absorbe el PC1—; estandarizado se necesitan **8**. Que 1 componente explique casi todo es una **señal de alarma**, no una buena noticia.

**Cómo se nombran los dos primeros ejes de Wine** (tabla de cargas que imprime la celda, hoja `cargas_wine`): **PC1 (36,2 % de la varianza) = riqueza fenólica** —`flavanoids` +0,92, `total_phenols` +0,86, `od280/od315` +0,82, `proanthocyanins` +0,68, `hue` +0,65, frente a `nonflavanoid_phenols` −0,65 y `malic_acid` −0,53—; **PC2 (19,2 %) = cuerpo / potencia física** —`color_intensity` +0,84, `alcohol` +0,77, `proline` +0,58—. ⚠️ La intuición enológica sugiere poner `alcohol` con los fenoles en PC1 y `hue` junto a `color_intensity` en PC2; **los datos dicen lo contrario**: `alcohol` carga +0,31 en PC1 y **+0,77 en PC2**, y `hue` carga **+0,65 en PC1** y −0,44 en PC2 (signo **opuesto** al de `color_intensity`). Nombrar un componente por lo que "debería" cargar en vez de por lo que carga es el error más frecuente del Drill 2.

**El paso opcional de alta dimensión (MNIST-784).** La muestra `mnist_muestra.npz` (2000 imágenes de 28×28 = **784** píxeles) responde la misma pregunta con 12 veces más dimensiones: hacen falta **82** de 784 componentes para el ≥90 %, una compresión de **9,6×** frente a los **3,0×** de `digits` (64 → 21). La lección escala: cuanto más alta es la dimensión nominal, mayor suele ser la redundancia y **más paga** comprimir —y ese ahorro se costea en la hoja `costeo_compresion`—. Se usa la muestra de ≤1 MB, no el MNIST completo (~55 MB), por la regla de datos del curso.

> El **scree plot** de Iris (figura de resultados) se genera en la **sección 6** leyendo el Excel, según la convención del curso.

## 6.8 — ¿Qué decisión habilita? Laboratorio: PCR sobre el conjunto de S04 (Sección 5 del cuaderno)

Tres piezas: **(a)** PCR/PLS contra la multicolinealidad de un dataset de negocio; **(b)** Análisis Factorial (nº de factores por Kaiser); **(c)** visualización con t-SNE/UMAP y su proxy cuantitativo. Guía paso a paso en `laboratorio/GUIA_LABORATORIO_S06.docx`.

### PCR y PLS contra la multicolinealidad (Communities and Crime) — capítulo 6.8 (subsección 5a)

Se reutiliza el dataset colineal de la Sesión 5: predecir la tasa de crímenes violentos a partir de indicadores socioeconómicos fuertemente correlacionados.

🔎 **Qué hace este código.** Carga Communities and Crime, prepara los predictores como en la Sesión 5 y mide su **multicolinealidad** (VIF de la inversa de la matriz de correlación).

In [ ]:
# Paso 1: cargar Communities and Crime y preparar los predictores (igual que en la Sesión 5)
crime = cargar_communities()
no_predictivas = ["state", "county", "community", "communityname", "fold"]
objetivo = "ViolentCrimesPerPop"
faltante = crime.isna().mean()
bloque_lemas = faltante[faltante > 0.5].index.tolist()      # bloque policial LEMAS (~84% faltante)
Xc = crime.drop(columns=no_predictivas + [objetivo] + bloque_lemas, errors="ignore")
Xc = Xc.fillna(Xc.mean(numeric_only=True))
yc = crime[objetivo].values
print(f"Predictores: {Xc.shape[1]}  |  observaciones: {Xc.shape[0]}")

# Multicolinealidad de los predictores ORIGINALES: VIF = diagonal de la inversa de la correlación
Xc_std = StandardScaler().fit_transform(Xc.values)
vif_orig = np.diag(np.linalg.pinv(np.corrcoef(Xc_std, rowvar=False)))
vif_orig_max = float(np.nanmax(vif_orig))
vif_orig_med = float(np.nanmedian(vif_orig))
n_vif_alto = int((vif_orig > 10).sum())
print(f"VIF de los predictores originales: máximo = {vif_orig_max:.1f}, mediana = {vif_orig_med:.0f}, "
      f"{n_vif_alto} de {Xc.shape[1]} con VIF > 10 (multicolinealidad severa)")

📖 **Cómo se lee.** Los predictores de negocio están **fuertemente correlacionados**: decenas superan un VIF de 10 (el máximo ronda ~1000). Es la multicolinealidad diagnosticada en la Sesión 4; la PCR/PLS la resolverá recombinando las variables en componentes ortogonales.

**❓ Qué se quiere averiguar.** Los 100 predictores de este dataset de negocio están tan correlacionados que el VIF máximo llega a ≈ 1000: los coeficientes del OLS son inestables e ininterpretables. ¿Cuánta capacidad predictiva cuesta cambiarlos por componentes ortogonales?

- **La decisión concreta:** si el precio en error resulta alto, conviene quedarse con el OLS y renunciar a interpretar. Si resulta bajo, se obtiene interpretabilidad casi sin costo, y esa es la segunda vía contra la multicolinealidad tras la regularización de la Sesión 5.
- **Antes de mirar el resultado:** si el MSE de prueba de la PCR **aumentara de forma pronunciada** frente al **0,01753** del OLS, recombinar habría destruido señal y la vía quedaría descartada. Si queda **al lado** de esa cifra, la multicolinealidad desaparece por construcción prácticamente sin costo. Queda una segunda hipótesis: la PLS mira la respuesta al construir sus componentes y la PCR no, así que ¿cuántos componentes necesita cada una para el mismo error?

🔎 **Qué hace este código.** Compara **OLS**, **PCR** y **PLS** (nº de componentes elegido por validación cruzada de 5 pliegues) sobre los mismos predictores, con `StandardScaler` **dentro** del `Pipeline` para que no haya fuga de información entre pliegues. Registra además tres resultados que antes se afirmaban sin cifra: la **curva completa de CV** (MSE de validación cruzada para cada *k* de la rejilla, no solo el mínimo), el **VIF de los scores de PLS** (que, como los del PCA, son ortogonales) y los **coeficientes de PLS** sobre los predictores estandarizados. Las tres salidas van al Excel (hojas `curva_cv_pcr`, `coeficientes_pls` y la columna `vif_maximo` de `comparacion_pcr`).

In [ ]:
# Paso 2: comparar OLS con PCR y PLS (nº de componentes elegido por validación cruzada)
Xtr, Xte, ytr, yte = train_test_split(Xc.values, yc, test_size=0.3, random_state=RANDOM_STATE)

def _mse_r2(modelo):
    modelo.fit(Xtr, ytr)
    pred = modelo.predict(Xte)
    return float(mean_squared_error(yte, pred)), float(r2_score(yte, pred))

def _cv_mse(modelo):
    return float(-cross_val_score(modelo, Xtr, ytr, cv=5, scoring="neg_mean_squared_error").mean())

mse_ols, r2_ols = _mse_r2(Pipeline([("s", StandardScaler()), ("m", LinearRegression())]))

# CURVA DE VALIDACIÓN CRUZADA: se calcula el MSE de CV para toda la rejilla y se elige el mínimo
# (la curva completa se registra en la hoja `curva_cv_pcr` del Excel; antes solo sobrevivía el argmin).
ks_pcr = [2, 5, 10, 15, 20, 30, 40, 50, 60, 80]
ks_pls = [2, 5, 10, 15, 20, 30]
cv_pcr = {k: _cv_mse(Pipeline([("s", StandardScaler()), ("p", PCA(n_components=k)),
                               ("m", LinearRegression())])) for k in ks_pcr}
cv_pls = {k: _cv_mse(Pipeline([("s", StandardScaler()), ("m", PLSRegression(n_components=k))]))
          for k in ks_pls}
k_pcr = min(ks_pcr, key=lambda k: cv_pcr[k])
k_pls = min(ks_pls, key=lambda k: cv_pls[k])
curva_cv = ([["PCR", k, float(cv_pcr[k])] for k in ks_pcr] +
            [["PLS", k, float(cv_pls[k])] for k in ks_pls])

mse_pcr, r2_pcr = _mse_r2(Pipeline([("s", StandardScaler()),
                                    ("p", PCA(n_components=k_pcr)), ("m", LinearRegression())]))
mse_pls, r2_pls = _mse_r2(Pipeline([("s", StandardScaler()), ("m", PLSRegression(n_components=k_pls))]))

# Los componentes del PCA son ortogonales -> su VIF es ≈ 1 (ya no hay multicolinealidad)
comp = PCA(n_components=k_pcr).fit_transform(Xc_std)
vif_comp_max = float(np.diag(np.linalg.pinv(np.corrcoef(comp, rowvar=False))).max())
# Los SCORES de PLS también son ortogonales -> su VIF también es ≈ 1. Se CALCULA aquí (antes se
# afirmaba sin cifra) y se registra en `comparacion_pcr`, columna vif_maximo, fila PLS.
sc_pls = PLSRegression(n_components=k_pls).fit(Xc_std, yc).x_scores_
vif_pls_max = float(np.diag(np.linalg.pinv(np.corrcoef(sc_pls, rowvar=False))).max())
# Coeficientes de PLS sobre los predictores estandarizados -> hoja `coeficientes_pls` del Excel.
_pipe_pls = Pipeline([("s", StandardScaler()), ("m", PLSRegression(n_components=k_pls))]).fit(Xtr, ytr)
coef_pls = np.asarray(_pipe_pls.named_steps["m"].coef_).ravel()
tabla_coef_pls = [[nom, float(c)] for nom, c in zip(Xc.columns, coef_pls)]

print(f"OLS          -> MSE prueba = {mse_ols:.5f} | R² = {r2_ols:.3f} | {Xc.shape[1]} predictores | VIF máx = {vif_orig_max:.1f}")
print(f"PCR (k={k_pcr:>2})  -> MSE prueba = {mse_pcr:.5f} | R² = {r2_pcr:.3f} | VIF de los componentes ≈ {vif_comp_max:.2f}")
print(f"PLS (k={k_pls:>2})  -> MSE prueba = {mse_pls:.5f} | R² = {r2_pls:.3f} | VIF de los scores ≈ {vif_pls_max:.2f}")
print(f"\nCurva de CV (MSE de validación cruzada por nº de componentes):")
print("  PCR  " + "  ".join(f"k={k}:{cv_pcr[k]:.5f}" for k in ks_pcr))
print("  PLS  " + "  ".join(f"k={k}:{cv_pls[k]:.5f}" for k in ks_pls))
print(f"  -> mínimo PCR en k={k_pcr}; mínimo PLS en k={k_pls}: la PLS llega al mismo error con "
      f"{k_pcr/k_pls:.0f} veces menos componentes porque mira la respuesta al construirlos.")
print(f"Los {len(tabla_coef_pls)} coeficientes de PLS (escala estandarizada) se registran en el Excel; "
      f"los 3 mayores en valor absoluto: "
      + ", ".join(f"{n} {c:+.4f}" for n, c in sorted(tabla_coef_pls, key=lambda r: -abs(r[1]))[:3]))

📖 **Lectura (interpretabilidad vs. multicolinealidad).** El OLS alcanza una precisión predictiva adecuada pero se apoya en **65 predictores con VIF > 10** (máximo ≈ 1000): coeficientes inestables e ininterpretables. La **PCR** logra un error de prueba comparable con componentes **ortogonales** cuyo **VIF ≈ 1** —la multicolinealidad diagnosticada en la Sesión 4 desaparece por construcción—. La **PLS**, al orientar los componentes hacia la respuesta, alcanza el mismo desempeño con **muchos menos componentes** que la PCR. Es la **segunda vía** contra la multicolinealidad (la primera fue la regularización de la Sesión 5).

> **No es clustering ni selección de variables:** la PCR **recombina** todas las variables en componentes; no descarta ninguna (a diferencia del LASSO).

### Análisis Factorial: nº de factores por Kaiser (Iris y Wine) — capítulo 6.4 (subsección 5b)

**❓ Qué se quiere averiguar.** Antes de buscar factores latentes hay una pregunta previa: ¿hay algo latente que buscar en estos datos, o el software devolverá factores igual aunque no lo haya?

- **Qué decide:** si el dato no es adecuado, las cargas rotadas que salgan no describen constructos —«cuerpo del vino», «perfil de color»—, sino ruido con nombre propio, y toda la interpretación posterior se apoya en nada. La regla de entrada de la sesión es doble: KMO ≥ 0,60 **y** Bartlett con p < 0,05.
- **Antes de mirar el resultado:** Wine trae 13 variables químicas correlacionadas y debería pasar el filtro; Iris trae solo 4 medidas y se factoriza aquí **a propósito** pese a no pasarlo. El riesgo se anuncia de antemano: la salida de Iris traerá una varianza acumulada alta y de buen aspecto, porque la librería no protesta nunca. La pregunta no es si el número se ve bien, sino **qué señal de la salida delata que no hay que creerlo** —una comunalidad por encima de 1 sería imposible, y bastaría para invalidar el ajuste.

🔎 **Qué hace este código.** Aplica Análisis Factorial (rotación varimax) a Iris y Wine, imprime el **KMO** y la **esfericidad de Bartlett** de cada uno junto al ajuste, decide el nº de factores por el criterio de **Kaiser** y guarda las **cargas factoriales rotadas** y las **comunalidades** (hojas `cargas_factoriales` y `adecuacion_af` del Excel).

> ⚠️ **CONTRAEJEMPLO DELIBERADO — Iris se factoriza a propósito violando su propio requisito.** La regla de entrada del Análisis Factorial en esta sesión es **doble**: KMO ≥ 0,60 **y** Bartlett con `p < 0,05` (la guía de supuestos de la sesión, Partes 2.1–2.3). Iris cumple Bartlett pero su **KMO es 0,5401**, por debajo del mínimo. Aun así se ajusta aquí, **a propósito**, para **ver qué pasa cuando el KMO no da**: qué salidas produce el software (porque produce salidas, y con apariencia convincente), y en qué se nota que no hay que creerlas. **NO se usaría en producción**, y ninguna decisión de la sesión se apoya en ese ajuste. Wine (KMO 0,7787) es el caso legítimo con el que se compara. La lectura de lo que aparece está en el bloque 📖 que sigue a la salida.

In [ ]:
# Análisis Factorial (varimax) sobre Iris y Wine: nº de factores por Kaiser y varianza acumulada
# ⚠️ CONTRAEJEMPLO DELIBERADO: Iris se factoriza A PROPÓSITO pese a que su KMO (0.5401) está por
# debajo del mínimo de 0.60 exigido por la sesión. Se hace para VER qué pasa cuando el KMO no da;
# NO se usaría en producción. Por eso el KMO se imprime AL LADO de cada ajuste: la violación queda
# a la vista en la misma línea que el resultado. Wine (KMO 0.7787) es el caso legítimo de contraste.
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

fa_resumen = {}
tabla_cargas_factoriales, tabla_adecuacion_af = [], []
for nombre, data, nf, cols in [("Iris", load_iris().data, 2, load_iris().feature_names),
                               ("Wine", load_wine().data, 3, load_wine().feature_names)]:
    Xs = StandardScaler().fit_transform(data)
    chi2_b, p_b = calculate_bartlett_sphericity(Xs)
    _, kmo = calculate_kmo(Xs)
    fa = FactorAnalyzer(n_factors=nf, rotation="varimax").fit(Xs)
    ev, _ = fa.get_eigenvalues()
    kaiser = int((ev > 1).sum())
    _, _, cum = fa.get_factor_variance()
    rotulo = ("CONTRAEJEMPLO deliberado: KMO < 0.60, NO se usaría en producción" if kmo < 0.60
              else "uso legítimo: KMO >= 0.60 y Bartlett p < 0.05")
    fa_resumen[nombre] = {"n_factores": nf, "kaiser": kaiser, "var_acumulada": float(cum[-1]),
                          "kmo": float(kmo), "rotulo": rotulo}
    tabla_adecuacion_af.append([nombre, float(kmo), float(chi2_b), float(p_b), kaiser, nf, rotulo])
    cargas_rot, comun = fa.loadings_, fa.get_communalities()
    for j, nm in enumerate(cols):
        tabla_cargas_factoriales.append([nombre, nm] + [float(cargas_rot[j, k]) for k in range(nf)]
                                        + [""] * (3 - nf) + [float(comun[j])])
    print(f"{nombre:5s} KMO = {kmo:.4f} "
          f"[{'ADECUADO (>=0.60)' if kmo >= 0.60 else 'POR DEBAJO DEL MÍNIMO 0.60'}] | "
          f"Bartlett chi2 = {chi2_b:.0f}, p = {p_b:.1e} | Kaiser (eigenvalor > 1) = {kaiser} factor(es) | "
          f"n_factores AJUSTADO = {nf} | varianza acumulada = {cum[-1]*100:.1f}%")
    print(f"      -> {rotulo}")
print("\nDos columnas distintas, no una: 'Kaiser = 1' (Iris) es lo que dice el CRITERIO; 'n_factores = 2'")
print("es lo que se AJUSTÓ a propósito para el contraejemplo. En el Excel viven separadas en la hoja")
print("`factor_analysis`: B = n_factores (elección del analista), C = kaiser_eig_gt1 (el criterio).")

📖 **Lectura — y qué se aprende del contraejemplo.** El **criterio de Kaiser** (eigenvalor > 1 sobre datos estandarizados) sugiere **1** factor común en Iris y **3** en Wine: coincide con la mayor complejidad química del vino. A diferencia del PCA (que describe la varianza total), el Análisis Factorial separa la **varianza común** (los factores) de la **única** (ruido de cada variable); tras varimax, cada factor se **nombra** por las variables que lo cargan. Detalle en `plantillas/guia_componentes_factores.docx`.

⚠️ **Qué pasó al factorizar Iris a propósito con KMO 0,5401 (< 0,60).** Tres señales, todas visibles en la salida y en la hoja `cargas_factoriales`:

1. **El software no protesta.** `FactorAnalyzer` devuelve dos factores con una varianza acumulada del **93,0 %**, una cifra que "se ve bien" y que un lector desprevenido citaría como éxito. La adecuación del dato no la juzga la librería: la juzga quien la usa, **antes** de ajustar.
2. **La solución es inestable (caso Heywood).** La comunalidad de `petal length` resulta **1,0106**, es decir **mayor que 1**: matemáticamente imposible, porque una comunalidad es la fracción de la varianza de la variable explicada por los factores comunes. Es el síntoma clásico de forzar factores donde no hay estructura latente que sostenga.
3. **El "2" no es de Kaiser.** El criterio da **1**; el 2 es una **elección del analista** hecha para este contraejemplo. Confundir las dos columnas (`n_factores` vs. `kaiser_eig_gt1`) convierte una decisión discrecional en un resultado objetivo que nadie calculó.

**Conclusión operativa:** Iris **no se factoriza** —la doble condición de entrada (KMO ≥ 0,60 **y** Bartlett `p < 0,05`) no se cumple— y nada en esta sesión se decide con ese ajuste. Wine (KMO 0,7787, Bartlett `p ≈ 2e−224`, Kaiser 3) es el caso legítimo: ahí sí se leen las cargas rotadas y se nombran los factores. El diagnóstico formal se repite, ya como prueba de supuestos, en la **Sección 8.2**.

### 6.5.3 — ¿Se reproduce su mapa de dígitos? Visualización con t-SNE y UMAP sobre digits y su proxy cuantitativo (subsección 5c)

Como t-SNE/UMAP producen un **mapa** (salida visual), se mide su calidad con un **proxy reproducible**: la exactitud de un **kNN(5)** entrenado sobre el embedding 2D y la **trustworthiness** (¿los vecinos del mapa lo eran en el original?).

**❓ Qué se quiere averiguar.** Un mapa 2D de dígitos siempre «se ve bien». ¿Cómo se comprueba con una cifra, y no con la vista, que ese mapa conservó la información de las 64 dimensiones originales?

- **Qué decide:** de esa cifra depende si el mapa sirve para sostener un hallazgo o solo para decorar una lámina. El techo lo pone el kNN(5) sobre los 64 originales: ninguna proyección puede superarlo por mérito propio.
- **Antes de mirar el resultado:** si el kNN sobre el mapa 2D cayera **cerca de 0,10**, el mapa no distinguiría dígitos y sería puro azar con diez clases. Si queda **junto al techo de 64 dimensiones**, la estructura local se conservó pese a la compresión. El PCA-2D sirve de contraste: solo retiene el 28,5 % de la varianza, así que debería perder mucho más. Y una cautela sobre el cuarto decimal: las cinco semillas que vienen después dirán cuánto de la diferencia entre t-SNE y UMAP es señal y cuánto es azar, antes de que alguien la lea como un ranking.

🔎 **Qué hace este código.** Construye los embeddings 2D de `digits` (PCA, t-SNE, UMAP) y mide su calidad con **kNN(5)** 5-fold y **trustworthiness** (semilla 42 para reproducibilidad). En t-SNE (openTSNE) se fijan dos argumentos que **cambian el resultado**: `initialization="pca"` (arranca desde el PCA en vez de al azar → capa determinista y mapa más estable) y `n_jobs=-1` (usa todos los núcleos: acelera, pero la reducción en paralelo hace que el mapa **no** sea reproducible bit-a-bit entre máquinas con distinto número de núcleos). Fijar `n_jobs=1` daría reproducibilidad exacta a costa de tiempo; por eso el material de referencia de la sesión cruza t-SNE/UMAP con **tolerancias**, no con igualdad exacta. Esta celda **sí** alimenta el Excel del contrato (celda 77) con su semilla fija.

Después repite el mismo ajuste con **cinco semillas** (0, 1, 7, 42, 2026) y recorre el **barrido de hiperparámetros del Drill 3** (t-SNE con `perplexity` 5/30/50; UMAP con cuatro combinaciones de `n_neighbors`/`min_dist`). Sirven para lo mismo: **poner una banda alrededor del cuarto decimal** antes de que alguien lo lea como un ranking. Ambas tablas se registran en el Excel (`estabilidad_semillas`, `barrido_drill3`). Es la parte más lenta del cuaderno (≈ 2 minutos: 13 ajustes de t-SNE/UMAP sobre 1797 imágenes).

In [ ]:
# Embeddings 2D de digits (64 dimensiones -> 2) y su proxy cuantitativo
Xf, yf = digits.data, digits.target
knn5 = lambda emb: float(cross_val_score(KNeighborsClassifier(5), np.asarray(emb), yf, cv=5).mean())

acc_full = knn5(Xf)                                          # techo: los 64 originales
emb_pca2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xf)
emb_tsne = np.asarray(TSNE_open(n_components=2, perplexity=30, initialization="pca",
                                random_state=RANDOM_STATE, n_jobs=-1).fit(Xf))
emb_umap = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                     random_state=RANDOM_STATE).fit_transform(Xf)

acc_pca2, tw_pca2 = knn5(emb_pca2), float(trustworthiness(Xf, emb_pca2, n_neighbors=5))
acc_tsne, tw_tsne = knn5(emb_tsne), float(trustworthiness(Xf, emb_tsne, n_neighbors=5))
acc_umap, tw_umap = knn5(emb_umap), float(trustworthiness(Xf, emb_umap, n_neighbors=5))

print(f"kNN(5) sobre 64D (techo)  = {acc_full:.4f}")
print(f"kNN(5) sobre PCA-2D       = {acc_pca2:.4f}   |  trustworthiness = {tw_pca2:.4f}")
print(f"kNN(5) sobre t-SNE-2D     = {acc_tsne:.4f}   |  trustworthiness = {tw_tsne:.4f}")
print(f"kNN(5) sobre UMAP-2D      = {acc_umap:.4f}   |  trustworthiness = {tw_umap:.4f}")

# --- ¿Cuánto de ese cuarto decimal es SEÑAL y cuánto es SEMILLA? (hoja `estabilidad_semillas`) ---
# Se repite el mismo ajuste con 5 semillas. El PCA-2D es determinista y sirve de control.
SEMILLAS = [0, 1, 7, 42, 2026]
tabla_semillas = []
for s in SEMILLAS:
    e_s = np.asarray(TSNE_open(n_components=2, perplexity=30, initialization="pca",
                               random_state=s, n_jobs=-1).fit(Xf))
    tabla_semillas.append(["t-SNE", s, knn5(e_s), float(trustworthiness(Xf, e_s, n_neighbors=5))])
    u_s = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=s).fit_transform(Xf)
    tabla_semillas.append(["UMAP", s, knn5(u_s), float(trustworthiness(Xf, u_s, n_neighbors=5))])
tabla_semillas.append(["PCA-2D", "deterministico", acc_pca2, tw_pca2])
_kt = [r[2] for r in tabla_semillas if r[0] == "t-SNE"]
_ku = [r[2] for r in tabla_semillas if r[0] == "UMAP"]
_gana_umap = sum(1 for _u, _t in zip(_ku, _kt) if _u > _t)
print(f"\nEstabilidad por semilla, kNN(5): t-SNE {min(_kt):.4f}-{max(_kt):.4f} (rango {max(_kt)-min(_kt):.4f}) | "
      f"UMAP {min(_ku):.4f}-{max(_ku):.4f} (rango {max(_ku)-min(_ku):.4f}) | PCA-2D {acc_pca2:.4f} (determinista)")
print(f"El valor de contrato de UMAP ({acc_umap:.4f}) es el MÍNIMO de las cinco semillas, y UMAP supera a")
print(f"t-SNE en {_gana_umap} de las 5: el orden 't-SNE > UMAP' NO es robusto y no debe enseñarse como")
print("un ranking. Lo que sí es robusto entre semillas: t-SNE y UMAP >> PCA-2D (0.60), y el orden de")
print("trustworthiness t-SNE > UMAP > PCA-2D.")

# --- Barrido de hiperparámetros del Drill 3 (semilla fija 42) -> hoja `barrido_drill3` ---
tabla_barrido = []
for _perp in [5, 30, 50]:
    e_b = np.asarray(TSNE_open(n_components=2, perplexity=_perp, initialization="pca",
                               random_state=RANDOM_STATE, n_jobs=-1).fit(Xf))
    tabla_barrido.append(["t-SNE", f"perplexity={_perp}", knn5(e_b),
                          float(trustworthiness(Xf, e_b, n_neighbors=5))])
for _nn, _md in [(5, 0.1), (15, 0.1), (50, 0.1), (15, 0.5)]:
    u_b = umap.UMAP(n_components=2, n_neighbors=_nn, min_dist=_md,
                    random_state=RANDOM_STATE).fit_transform(Xf)
    tabla_barrido.append(["UMAP", f"n_neighbors={_nn}, min_dist={_md}", knn5(u_b),
                          float(trustworthiness(Xf, u_b, n_neighbors=5))])
print("\nBarrido del Drill 3 (semilla 42), kNN(5):")
for _tec, _par, _k, _tw in tabla_barrido:
    print(f"  {_tec:6s} {_par:30s} kNN = {_k:.4f}   trustworthiness = {_tw:.4f}")
print(f"La fila canónica (t-SNE perplexity=30) da {tabla_barrido[1][2]:.4f}, el mismo valor que alimenta")
print("el Excel de contrato: la tabla del Drill 3 y el ancla de la sesión son el MISMO número.")

🔎 **Qué hace este código.** Dibuja los mapas 2D de t-SNE y UMAP coloreados por dígito (figuras de **visualización**, por cálculo directo).

In [ ]:
# Mapas 2D de digits coloreados por dígito (figuras de VISUALIZACIÓN -> cálculo directo)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), constrained_layout=True)
for ax, emb, titulo, acc in [(axes[0], emb_tsne, "t-SNE", acc_tsne),
                             (axes[1], emb_umap, "UMAP", acc_umap)]:
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=yf, cmap="tab10", s=8, alpha=0.85)
    ax.set_title(f"{titulo} 2D de digits  (kNN(5) = {acc:.2f})")
    ax.set_xlabel("dimensión 1"); ax.set_ylabel("dimensión 2")
fig.colorbar(sc, ax=axes, label="dígito", ticks=range(10))
mostrar(fig, FIGURAS / "S06_embeddings_digits.png")

📖 **Lectura (y límites).** t-SNE y UMAP **separan los 10 dígitos** en agrupaciones: recuperan en 2D casi toda la señal (kNN ≈ 0.97; trustworthiness ≈ 0.99), mientras el **PCA-2D** pierde la mitad (kNN ≈ 0.5–0.6, solo retiene 28,5 % de la varianza). Una exactitud alta prueba que se preservó la **estructura local** —**no** que las distancias del mapa sean exactas—. **Lo que NO se debe leer:** la distancia entre agrupaciones, el tamaño de una agrupación o su densidad. Y esto **no es clustering**: solo se visualiza y se mide la calidad del embedding (agrupar formalmente es la Sesión 7).

## Transversal — Exportación a Excel y figuras de resultados (Sección 6 del cuaderno)

Convención del curso: los resultados se vuelcan a `resultados/S06_resultados.xlsx` (openpyxl) y las **figuras de resultados se generan LEYENDO ese Excel**. Las figuras de visualización (biplot, scatter t-SNE/UMAP) ya se trazaron por cálculo directo.

🔎 **Qué hace este código.** Vuelca los resultados a las **catorce** hojas del **contrato** en `S06_resultados.xlsx`. Es la **única** celda que escribe el Excel.

| Hoja | Contenido |
|---|---|
| `reduccion_pca` | Contrato A1:B5: Iris PC1/PC2 por covarianza, digits ≥90 %, kNN de UMAP |
| `varianza_explicada` | Iris cov/corr, Wine crudo/estandarizado y digits: explicada y acumulada |
| `factor_analysis` | Iris/Wine: `n_factores` ajustado, `kaiser_eig_gt1`, varianza acumulada (+ columnas que declaran cuál es cuál) |
| `embeddings` | kNN(5) y trustworthiness por representación (64D, PCA-2D, t-SNE, UMAP) |
| `comparacion_pcr` | OLS vs. PCR vs. PLS: nº de componentes, MSE, R² y **VIF máximo (las tres filas)** |
| `cargas_wine` | Las 13 variables de Wine estandarizado en PC1 y PC2: autovector y carga |
| `cargas_biplot_iris` | Iris por covarianza: autovector, carga `v·√λ` (en cm), σ y correlación |
| `cargas_factoriales` | Cargas rotadas (varimax) y comunalidades de Iris y Wine |
| `adecuacion_af` | KMO, Bartlett y Kaiser con el veredicto de uso (incluye el contraejemplo de Iris) |
| `curva_cv_pcr` | MSE de validación cruzada por nº de componentes (PCR y PLS), con el mínimo marcado |
| `coeficientes_pls` | Los 100 coeficientes de PLS sobre predictores estandarizados |
| `estabilidad_semillas` | kNN y trustworthiness de t-SNE/UMAP con 5 semillas (PCA-2D como control) |
| `barrido_drill3` | Barrido de `perplexity` (t-SNE) y `n_neighbors`/`min_dist` (UMAP) del Drill 3 |
| `costeo_compresion` | La compresión traducida a factura: GB y $/mes por millón de vectores, con su ahorro |

Las cinco primeras hojas conservan **verbatim** los valores del contrato histórico; las **nueve** restantes registran lo que hasta ahora se citaba en los materiales sin que ninguna celda lo produjera. `costeo_compresion` es el **único** costeo de la sesión: la guía del docente y el entregable citan **esta** hoja y ninguna cifra propia.

In [ ]:
# Construir el Excel con las hojas del contrato de la sesión
from openpyxl import Workbook
wb = Workbook()

# --- Hoja 1: reduccion_pca (CONTRATO EXACTO A1:B5; valores CALCULADOS, ratios 0-1) ---
ws = wb.active; ws.title = "reduccion_pca"
ws["A1"] = "metrica";            ws["B1"] = "valor"
ws["A2"] = "iris_pc1_cov";       ws["B2"] = round(iris_pc1_cov, 4)
ws["A3"] = "iris_pc2_cov";       ws["B3"] = round(iris_pc2_cov, 4)
ws["A4"] = "digits_n_comp_90";   ws["B4"] = int(digits_n90)
ws["A5"] = "embedding_knn_umap"; ws["B5"] = round(acc_umap, 4)

# --- Hoja 2: varianza_explicada (Iris cov/corr PC1-4; Wine crudo/std; digits acumulada) ---
ws2 = wb.create_sheet("varianza_explicada")
ws2.append(["dataset", "convencion", "componente", "varianza_explicada", "varianza_acumulada"])
def _volcar(nombre, conv, evr):
    ac = np.cumsum(evr)
    for i, v in enumerate(evr):
        ws2.append([nombre, conv, i + 1, round(float(v), 6), round(float(ac[i]), 6)])
_volcar("Iris", "covarianza",  iris_cov_evr)
_volcar("Iris", "correlacion", iris_cor_evr)
_volcar("Wine", "covarianza",  PCA().fit(wine.data).explained_variance_ratio_)
_volcar("Wine", "correlacion", PCA().fit(StandardScaler().fit_transform(wine.data)).explained_variance_ratio_)
_volcar("digits", "covarianza", digits_evr)

# --- Hoja 3: factor_analysis (Iris y Wine: nº de factores por Kaiser y varianza acumulada) ---
# A:D se conservan VERBATIM (contrato histórico). E y F se AÑADEN para que no quepa ambigüedad
# sobre qué columna es el criterio y cuál la elección del analista (B NO es Kaiser; C sí lo es).
ws3 = wb.create_sheet("factor_analysis")
ws3.append(["dataset", "n_factores", "kaiser_eig_gt1", "var_acumulada",
            "origen_de_n_factores", "nota_lectura"])
for nombre in ["Iris", "Wine"]:
    r = fa_resumen[nombre]
    origen = (f"ELECCION DEL ANALISTA (nf={r['n_factores']} ajustado a mano); el criterio de Kaiser "
              f"da {r['kaiser']}")
    nota = (f"columna B = n_factores AJUSTADO, NO es Kaiser | columna C = kaiser_eig_gt1 = "
            f"{r['kaiser']} | KMO = {r['kmo']:.4f} -> {r['rotulo']}")
    ws3.append([nombre, r["n_factores"], r["kaiser"], round(r["var_acumulada"], 4), origen, nota])

# --- Hoja 4: embeddings (kNN(5) y trustworthiness sobre digits) ---
ws4 = wb.create_sheet("embeddings")
ws4.append(["representacion", "knn5", "trustworthiness"])
ws4.append(["64D (original)", round(acc_full, 4), ""])
ws4.append(["PCA-2D",   round(acc_pca2, 4), round(tw_pca2, 4)])
ws4.append(["t-SNE-2D", round(acc_tsne, 4), round(tw_tsne, 4)])
ws4.append(["UMAP-2D",  round(acc_umap, 4), round(tw_umap, 4)])

# --- Hoja extra: comparacion_pcr (OLS vs PCR vs PLS en Communities) ---
ws5 = wb.create_sheet("comparacion_pcr")
ws5.append(["modelo", "n_componentes", "mse_prueba", "r2_prueba", "vif_maximo"])
ws5.append(["OLS", int(Xc.shape[1]), round(mse_ols, 5), round(r2_ols, 4), round(vif_orig_max, 1)])
ws5.append(["PCR", int(k_pcr),       round(mse_pcr, 5), round(r2_pcr, 4), round(vif_comp_max, 2)])
ws5.append(["PLS", int(k_pls),       round(mse_pls, 5), round(r2_pls, 4), round(vif_pls_max, 2)])

# --- Hoja 6: cargas_wine (las 13 variables de Wine ESTANDARIZADO en PC1 y PC2) ---
# Convención: carga = v·√λ. Sobre datos estandarizados (matriz de correlación) la carga ES la
# correlación variable-componente, acotada por 1. Fuente del nombramiento de los ejes del Drill 2.
ws6 = wb.create_sheet("cargas_wine")
ws6.append(["variable", "componente_pc1", "carga_pc1", "componente_pc2", "carga_pc2"])
for nom, v1, c1, v2, c2 in tabla_cargas_wine:
    ws6.append([nom, round(v1, 4), round(c1, 4), round(v2, 4), round(c2, 4)])

# --- Hoja 7: cargas_biplot_iris (Iris CRUDO, convención de COVARIANZA) ---
# Tres escalas distintas de la misma flecha: autovector unitario, carga v·√λ (en cm) y correlación
# (v·√λ/σ). La columna de control es la correlación EMPÍRICA corr(variable, score), calculada aparte.
ws7 = wb.create_sheet("cargas_biplot_iris")
ws7.append(["variable", "componente_pc1", "carga_vraizlambda_pc1_unidades", "sigma_variable",
            "correlacion_pc1", "correlacion_empirica_pc1", "componente_pc2",
            "carga_vraizlambda_pc2_unidades", "correlacion_pc2", "correlacion_empirica_pc2"])
for fila in tabla_cargas_biplot_iris:
    ws7.append([fila[0]] + [round(float(x), 4) for x in fila[1:]])

# --- Hoja 8: cargas_factoriales (varimax) + comunalidades ---
ws8 = wb.create_sheet("cargas_factoriales")
ws8.append(["dataset", "variable", "factor1", "factor2", "factor3", "comunalidad"])
for fila in tabla_cargas_factoriales:
    ws8.append([fila[0], fila[1]] + [round(x, 4) if isinstance(x, float) else x for x in fila[2:]])

# --- Hoja 9: adecuacion_af (KMO, Bartlett, Kaiser y rótulo de contraejemplo) ---
ws9 = wb.create_sheet("adecuacion_af")
ws9.append(["dataset", "kmo", "bartlett_chi2", "bartlett_p", "kaiser_eig_gt1",
            "n_factores_ajustado", "veredicto_de_uso"])
for nombre, kmo, chi2_b, p_b, kaiser, nf, rotulo in tabla_adecuacion_af:
    ws9.append([nombre, round(kmo, 4), round(chi2_b, 1), float(f"{p_b:.3e}"), kaiser, nf, rotulo])

# --- Hoja 10: curva_cv_pcr (MSE de validación cruzada vs nº de componentes, PCR y PLS) ---
ws10 = wb.create_sheet("curva_cv_pcr")
ws10.append(["modelo", "n_componentes", "mse_cv_5fold", "es_minimo"])
for modelo, k, mse_cv in curva_cv:
    ws10.append([modelo, int(k), round(mse_cv, 5),
                 int((modelo == "PCR" and k == k_pcr) or (modelo == "PLS" and k == k_pls))])

# --- Hoja 11: coeficientes_pls (predictores estandarizados, k elegido por CV) ---
ws11 = wb.create_sheet("coeficientes_pls")
ws11.append(["variable", f"coef_pls_k{int(k_pls)}_estandarizado"])
for nom, c in tabla_coef_pls:
    ws11.append([nom, round(c, 6)])

# --- Hoja 12: estabilidad_semillas (5 semillas x t-SNE/UMAP; PCA-2D como control determinista) ---
ws12 = wb.create_sheet("estabilidad_semillas")
ws12.append(["tecnica", "semilla", "knn5", "trustworthiness"])
for tecnica, semilla, k5, tw in tabla_semillas:
    ws12.append([tecnica, semilla, round(k5, 4), round(tw, 4)])

# --- Hoja 13: barrido_drill3 (hiperparámetros de t-SNE y UMAP, semilla 42) ---
ws13 = wb.create_sheet("barrido_drill3")
ws13.append(["tecnica", "hiperparametros", "knn5", "trustworthiness"])
for tecnica, params, k5, tw in tabla_barrido:
    ws13.append([tecnica, params, round(k5, 4), round(tw, 4)])

# --- Hoja 14: costeo_compresion (la compresión, traducida a factura) ---
# ÚNICO costeo de la sesión: guía del docente y entregable citan ESTA hoja, no cifras propias.
# Todo sale de los datos de la sesión salvo UN parámetro declarado ILUSTRATIVO: la tarifa de
# almacenamiento de un vector store gestionado, 0,25 $/GB-mes, la MISMA para las tres filas.
TARIFA_USD_GB_MES, N_VECTORES = 0.25, 1_000_000

def _fila_costeo(escenario, dim0, dim1, bytes0, bytes1):
    "GB y $/mes de un índice de N_VECTORES antes y después de comprimir."
    gb0 = N_VECTORES * dim0 * bytes0 / 1e9
    gb1 = N_VECTORES * dim1 * bytes1 / 1e9
    c0, c1 = gb0 * TARIFA_USD_GB_MES, gb1 * TARIFA_USD_GB_MES
    return [escenario, int(dim0), int(dim1), int(bytes0), int(bytes1), round(gb0, 4),
            round(gb1, 4), round(gb0 / gb1, 3), TARIFA_USD_GB_MES, round(c0, 4), round(c1, 4),
            round(c0 - c1, 4)]

tabla_costeo = [
    _fila_costeo("indice RAG 1024d float32 -> 512d float8 (PCA 50% + cuantizacion; arXiv 2505.00105)",
                 1024, 512, 4, 1),
    _fila_costeo(f"digits 64 px -> {digits_n90} componentes (>=90% varianza; celda 54)",
                 64, digits_n90, 4, 4),
    _fila_costeo(f"Wine 13 variables -> {wine_n90_std} componentes (>=90% varianza; celda 56)",
                 13, wine_n90_std, 4, 4),
]
ws14 = wb.create_sheet("costeo_compresion")
ws14.append(["escenario", "dim_origen", "dim_destino", "bytes_valor_origen", "bytes_valor_destino",
             "gb_origen", "gb_destino", "factor_compresion", "tarifa_usd_gb_mes",
             "costo_origen_usd_mes", "costo_destino_usd_mes", "ahorro_usd_mes"])
for _f in tabla_costeo:
    ws14.append(_f)

wb.save(XLSX)
print("Excel guardado en:", XLSX)
print(f"Hojas ({len(wb.sheetnames)}):", wb.sheetnames)
print("\nCosteo de la compresión (tarifa ÚNICA 0,25 $/GB-mes; 1 000 000 de vectores):")
for _f in tabla_costeo:
    print(f"  {_f[0][:62]:62s} {_f[5]:8.4f} GB -> {_f[6]:8.4f} GB  ({_f[7]:.3f}x)  "
          f"{_f[9]:.4f} -> {_f[10]:.4f} $/mes  | ahorro {_f[11]:.4f} $/mes")

🔎 **Qué hace este código.** Lee la hoja `varianza_explicada` del Excel y dibuja el **scree plot** de Iris (convención de covarianza).

In [ ]:
# FIGURA DE RESULTADOS 1 (leyendo el Excel): scree plot de Iris (convención de covarianza)
ve = pd.read_excel(XLSX, sheet_name="varianza_explicada")
scree = ve[(ve["dataset"] == "Iris") & (ve["convencion"] == "covarianza")].sort_values("componente")
fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.plot(scree["componente"], scree["varianza_explicada"] * 100, "o-", color=UPC_ROJO, lw=2)
for x, y in zip(scree["componente"], scree["varianza_explicada"] * 100):
    ax.text(x, y + 3, f"{y:.1f}%", ha="center", fontsize=9, color=UPC_TINTA)
ax.set_xlabel("componente principal"); ax.set_ylabel("varianza explicada (%)")
ax.set_xticks(scree["componente"]); ax.set_ylim(0, 100)
ax.set_title("Scree plot de Iris: el codo tras PC1-PC2 marca cuántos retener")
mostrar(fig, FIGURAS / "S06_scree_iris.png")

🔎 **Qué hace este código.** Lee la hoja `embeddings` del Excel y compara la exactitud **kNN(5)** de cada representación en una barra.

In [ ]:
# FIGURA DE RESULTADOS 2 (leyendo el Excel): kNN(5) por representación (digits en 2D)
emb_tab = pd.read_excel(XLSX, sheet_name="embeddings")
fig, ax = plt.subplots(figsize=(7.2, 4.2))
colores = [UPC_GRIS, "#E4879C", UPC_ROJO, UPC_TINTA]
barras = ax.bar(emb_tab["representacion"], emb_tab["knn5"], color=colores)
for b, v in zip(barras, emb_tab["knn5"]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.015, f"{v:.2f}", ha="center", fontsize=10)
ax.set_ylabel("exactitud kNN(5), 5-fold"); ax.set_ylim(0, 1.08)
ax.set_title("t-SNE y UMAP recuperan en 2D casi toda la señal; el PCA-2D pierde la mitad")
mostrar(fig, FIGURAS / "S06_comparacion_embeddings.png")

🔎 **Qué hace este código.** **FIGURA DE RESULTADOS 3.** Lee la hoja `curva_cv_pcr` del Excel y **traza la curva de error de validación cruzada** de la PCR y de la PLS frente al número de componentes, marcando el mínimo de cada una y el **codo** de la PCR. Al lado, resuelve en una sola tabla y en un solo gráfico el **contraste con la Sesión 5**: el MSE de **prueba** de OLS / Ridge / LASSO / ElasticNet (hoja `communities` del Excel de S05, mismo dataset y mismo split) frente al de PCR / PLS (hoja `comparacion_pcr` de esta sesión). Es exactamente lo que el criterio **C4** del entregable (4 puntos) exige: curva **trazada** y error **contrastado en una misma tabla**. ⚠️ Los dos paneles miden aspectos distintos —el izquierdo, error de **CV** sobre el train; el derecho, error de **prueba**—: no se mezclan en el mismo eje.


In [ ]:
# FIGURA DE RESULTADOS 3 (leyendo el Excel): curva de CV de PCR/PLS + contraste con S05
curva = pd.read_excel(XLSX, sheet_name="curva_cv_pcr")
comp_s06 = pd.read_excel(XLSX, sheet_name="comparacion_pcr")

# --- (a) curva de CV: el error de validación cruzada frente al nº de componentes ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.4, 4.6))
for modelo, color in [("PCR", UPC_ROJO), ("PLS", UPC_TINTA)]:
    d = curva[curva["modelo"] == modelo].sort_values("n_componentes")
    ax1.plot(d["n_componentes"], d["mse_cv_5fold"], "o-", color=color, lw=2, label=modelo)
    m = d[d["es_minimo"] == 1].iloc[0]
    ax1.scatter([m["n_componentes"]], [m["mse_cv_5fold"]], s=170, facecolors="none",
                edgecolors=color, lw=2.2, zorder=5)
    ax1.annotate(f"mínimo {modelo}: k={int(m['n_componentes'])}\n{m['mse_cv_5fold']:.5f}",
                 (m["n_componentes"], m["mse_cv_5fold"]), textcoords="offset points",
                 xytext=(-6, 22 if modelo == "PCR" else -34), ha="center", fontsize=9, color=color)
d_pcr = curva[curva["modelo"] == "PCR"].sort_values("n_componentes")
codo = d_pcr[d_pcr["n_componentes"] == 15].iloc[0]
ax1.axvline(15, color=UPC_GRIS, ls="--", lw=1.2)
ax1.text(16, d_pcr["mse_cv_5fold"].max() * 0.93, f"codo de la PCR (k=15)\n{codo['mse_cv_5fold']:.5f}",
         fontsize=9, color=UPC_GRIS)
ax1.set_xlabel("nº de componentes (k)"); ax1.set_ylabel("MSE de validación cruzada (5-fold)")
ax1.set_title("Así se elige k: por la curva de CV, no a ojo"); ax1.legend()

# --- (b) contraste de ERROR DE PRUEBA con las vías de S04/S05 (mismo dataset y mismo split) ---
S05_XLSX = SESION.parent / "S05_regularizacion" / "resultados" / "S05_resultados.xlsx"
filas_cmp = [(r["modelo"], float(r["mse_prueba"]), f"{int(r['n_componentes'])} comp.", "S06")
             for _, r in comp_s06.iterrows() if r["modelo"] != "OLS"]
if S05_XLSX.exists():
    s05 = pd.read_excel(S05_XLSX, sheet_name="communities")
    filas_s05 = [(r["modelo"], float(r["mse_test"]), f"{int(r['n_variables_seleccionadas'])} vars.",
                  "S05") for _, r in s05.iterrows()]
else:
    print("aviso: no se encontró el Excel de S05; el panel derecho muestra solo las vías de S06.")
    filas_s05 = [("OLS", float(comp_s06.loc[comp_s06["modelo"] == "OLS", "mse_prueba"].iloc[0]),
                  "100 vars.", "S06")]
comparada = pd.DataFrame(filas_s05 + filas_cmp,
                         columns=["modelo", "mse_prueba", "complejidad", "sesion"])
col_bar = [UPC_GRIS if s == "S05" else UPC_ROJO for s in comparada["sesion"]]
bars = ax2.bar(comparada["modelo"], comparada["mse_prueba"], color=col_bar)
for b, v, c in zip(bars, comparada["mse_prueba"], comparada["complejidad"]):
    ax2.text(b.get_x() + b.get_width()/2, v + 0.00012, f"{v:.5f}\n{c}", ha="center", fontsize=8.5)
ax2.set_ylim(0, comparada["mse_prueba"].max() * 1.28)
ax2.set_ylabel("MSE de prueba")
ax2.set_title("Ninguna vía gana en predicción: cambia QUÉ se gana")
mostrar(fig, FIGURAS / "S06_curva_cv_pcr.png")

print("TABLA COMPARADA (mismo dataset Communities and Crime, mismo split, random_state=42):")
print(comparada.assign(via=["regularización (S05)" if s == "S05" else "componentes (S06)"
                            for s in comparada["sesion"]]).to_string(index=False))
print("\nLectura: el rango completo de MSE va de 0.01744 (Ridge) a 0.01847 (PCR) — menos de un 6 %")
print("de diferencia. La regularización CONSERVA variables con nombre y selecciona (LASSO deja 62")
print("de 100); la PCR/PLS RECOMBINA y mata la multicolinealidad (VIF 1037.4 -> 1) a costa del")
print("nombre. Se elige por lo que se necesita —interpretar variables o estabilizar la estimación—,")
print("no por el cuarto decimal del MSE.")


🔎 **Qué hace este código.** **FIGURA DE RESULTADOS 4 — el biplot de Wine.** Lee las **26 cargas** de la hoja `cargas_wine` del Excel (columnas `carga_pc1`/`carga_pc2`, la convención `v·√λ`) y las dibuja como flechas sobre los *scores* de PC1-PC2 de **Wine estandarizado**, con los puntos coloreados por cultivar. Es el biplot que califican el **criterio C2** del entregable y el **Drill 2**: aquí queda **trazado y nombrado** (PC1 = riqueza fenólica, PC2 = cuerpo / potencia física) para que el alumno lo reproduzca sobre su propio análisis. Como Wine está estandarizado, estas cargas **sí** son correlaciones y viven en [−1, 1] —a diferencia del biplot de Iris por covarianza de la celda 51—.


In [ ]:
# FIGURA DE RESULTADOS 4 (leyendo el Excel): biplot de Wine estandarizado (hoja `cargas_wine`)
cw = pd.read_excel(XLSX, sheet_name="cargas_wine")
Zw = pca_wine_std.transform(Xw_std)[:, :2]              # scores de PC1-PC2 (178 vinos)
ve_w = pca_wine_std.explained_variance_ratio_[:2]
esc_w = np.abs(Zw).max(axis=0) * 0.92                   # las cargas están en [-1, 1]

fig, ax = plt.subplots(figsize=(8.6, 6.6))
for k, nombre_c in enumerate(["cultivar 0", "cultivar 1", "cultivar 2"]):
    m = wine.target == k
    ax.scatter(Zw[m, 0], Zw[m, 1], s=22, alpha=0.65, color=PALETA[k], label=nombre_c)
for _, r in cw.iterrows():
    x, y = float(r["carga_pc1"]) * esc_w[0], float(r["carga_pc2"]) * esc_w[1]
    ax.arrow(0, 0, x, y, color=UPC_TINTA, width=0.008, head_width=0.075,
             length_includes_head=True, alpha=0.85)
    ax.text(x * 1.12, y * 1.12, str(r["variable"]), color=UPC_TINTA, fontsize=8.2, ha="center",
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.72))
ax.axhline(0, color=UPC_GRIS, lw=0.6); ax.axvline(0, color=UPC_GRIS, lw=0.6)
ax.set_xlabel(f"PC1 ({ve_w[0]*100:.1f}% de la varianza) — riqueza fenólica")
ax.set_ylabel(f"PC2 ({ve_w[1]*100:.1f}%) — cuerpo / potencia física")
ax.set_title("Biplot de Wine estandarizado: las flechas salen de la hoja `cargas_wine`")
ax.legend(title="cultivar", loc="lower right")
mostrar(fig, FIGURAS / "S06_biplot_wine.png")

_top1 = cw.reindex(cw["carga_pc1"].abs().sort_values(ascending=False).index).head(3)
_top2 = cw.reindex(cw["carga_pc2"].abs().sort_values(ascending=False).index).head(3)
print("Flechas leídas del Excel (hoja `cargas_wine`), no recalculadas aquí.")
print("PC1, |carga| más alta: " + ", ".join(f"{r.variable} {r.carga_pc1:+.4f}" for r in _top1.itertuples()))
print("PC2, |carga| más alta: " + ", ".join(f"{r.variable} {r.carga_pc2:+.4f}" for r in _top2.itertuples()))
print("Cómo se lee: ángulo ~0° entre flechas = variables que viajan juntas; ~180° = opuestas;")
print("~90° = casi sin correlación. Flecha corta = variable mal representada en ESTE plano (los")
print("dos ejes solo recogen el " + f"{ve_w.sum()*100:.1f}" + "% de la varianza de las 13 variables).")
print("OJO (Drill 2): `alcohol` NO está en el PC1 (+0.3140) sino en el PC2 (+0.7664), y `hue` SÍ")
print("está en el PC1 (+0.6455) con signo NEGATIVO en el PC2 (-0.4425), opuesto a color_intensity.")


🔎 **Qué hace este código.** Confronta los resultados obtenidos contra los **targets** de la réplica (sin `assert`; la validación con tolerancias vive en QA).

In [ ]:
# Comparación con los targets de la réplica (la ficha de réplica del paper) — SIN asserts
print(f"{'Resultado':40s}{'esperado':>12s}{'obtenido':>12s}")
print("-" * 64)
for nombre, esperado, obtenido in [
    ("Iris PC1 (covarianza)",        "0.9246", f"{iris_pc1_cov:.4f}"),
    ("Iris PC2 (covarianza)",        "0.0531", f"{iris_pc2_cov:.4f}"),
    ("Iris PC1 (correlacion)",       "0.7296", f"{iris_pc1_cor:.4f}"),
    ("Iris PC2 (correlacion)",       "0.2285", f"{iris_pc2_cor:.4f}"),
    ("Kaiser Iris (correlacion)",    "1",      f"{iris_kaiser}"),
    ("Wine estandarizado >=90% var", "8",      f"{wine_n90_std}"),
    ("digits >=90% var",             "21",     f"{digits_n90}"),
    ("kNN UMAP-2D (>> PCA-2D)",      ">=0.90", f"{acc_umap:.4f}"),
]:
    print(f"{nombre:40s}{esperado:>12s}{obtenido:>12s}")

📖 **Lectura de las figuras de resultados.** El **scree plot** de Iris muestra un codo nítido tras PC1-PC2 (92 %→5 %→2 %→0.5 %): dos componentes bastan. La **barra comparativa** de embeddings confirma el proxy: t-SNE y UMAP casi igualan el techo de 64D, mientras el PCA-2D queda muy por debajo. Todos los targets caen dentro de tolerancia (la validación formal con `assert` vive en el material de referencia de la sesión).

## Transversal — ✅ Verificación desde la base (Sección 6.1 del cuaderno)

Antes de confiar en el Excel, se **recomputan** los resultados clave desde los datos ya cargados y se cruzan con lo que se acaba de escribir (`assert recomputado ≈ Excel`). Esta subsección **no toca** el Excel: solo lo lee para comprobar que refleja la ejecución, no un registro tecleado.

🔎 **Qué hace este código.** Recomputa desde los datos las razones de varianza (cov/corr) y el nº de componentes ≥90 % en `digits`, y **reajusta t-SNE y UMAP de forma INDEPENDIENTE desde `digits.data`** (objetos nuevos, semilla 42 — **no** reutiliza los embeddings ya calculados en memoria) para medir su trustworthiness. Luego cruza todo con lo escrito en el Excel (`assert`): el PCA (determinista) con tolerancia estricta (0.01) y los embeddings estocásticos con tolerancia **holgada** (0.02) por el no determinismo de openTSNE/UMAP.

In [ ]:
# ✅ Recomputar los resultados clave desde los datos y cruzarlos con el Excel ya escrito
# (esta celda NO escribe el Excel: solo lo lee para verificar que es producto de la ejecucion).
# GENUINA, no tautologica: los embeddings estocasticos (t-SNE/UMAP) se REAJUSTAN AQUI de forma
# INDEPENDIENTE desde digits.data (objetos nuevos, misma semilla 42), NO se reutilizan emb_tsne/emb_umap
# en memoria (los de la celda 71 que alimentaron el Excel). Por el no determinismo de t-SNE/UMAP
# (openTSNE con n_jobs=-1 paraleliza segun nucleos) la tolerancia de esas dos filas es HOLGADA (0.02);
# la del PCA, determinista, se mantiene estricta (0.01).
evr_iris_cov = PCA().fit(X_iris).explained_variance_ratio_
# --- reajuste INDEPENDIENTE de t-SNE y UMAP desde la base (misma logica que el material de referencia de la sesión) ---
emb_tsne_v = np.asarray(TSNE_open(n_components=2, perplexity=30, initialization="pca",
                                  random_state=RANDOM_STATE, n_jobs=-1).fit(Xf))
emb_umap_v = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                       random_state=RANDOM_STATE).fit_transform(Xf)
rc = {
    "iris_pc1_cov": float(evr_iris_cov[0]),
    "iris_pc2_cov": float(evr_iris_cov[1]),
    "iris_pc1_cor": float(PCA().fit(StandardScaler().fit_transform(X_iris)).explained_variance_ratio_[0]),
    "digits_n90":   int(np.argmax(np.cumsum(PCA().fit(digits.data).explained_variance_ratio_) >= 0.90) + 1),
    "tw_tsne":      float(trustworthiness(Xf, emb_tsne_v, n_neighbors=5)),   # embedding REAJUSTADO
    "tw_umap":      float(trustworthiness(Xf, emb_umap_v, n_neighbors=5)),   # embedding REAJUSTADO
}

# leer de vuelta el Excel del contrato
pca_x = pd.read_excel(XLSX, sheet_name="reduccion_pca").set_index("metrica")["valor"]
ve_x  = pd.read_excel(XLSX, sheet_name="varianza_explicada")
emb_x = pd.read_excel(XLSX, sheet_name="embeddings").set_index("representacion")
m = (ve_x.dataset == "Iris") & (ve_x.convencion == "correlacion") & (ve_x.componente == 1)
excel_pc1_cor = float(ve_x[m]["varianza_explicada"].iloc[0])

# (nombre, recomputado, paper/ref, Excel, tolerancia): PCA estricta 0.01; embeddings holgada 0.02
filas = [
    ("Iris PC1 covarianza",   rc["iris_pc1_cov"], 0.9246, float(pca_x["iris_pc1_cov"]),                     0.01),
    ("Iris PC2 covarianza",   rc["iris_pc2_cov"], 0.0531, float(pca_x["iris_pc2_cov"]),                     0.01),
    ("Iris PC1 correlacion",  rc["iris_pc1_cor"], 0.7296, excel_pc1_cor,                                    0.01),
    ("digits n comp >=90%",   rc["digits_n90"],   21,     float(pca_x["digits_n_comp_90"]),                 0.5),
    ("trustworthiness t-SNE", rc["tw_tsne"],      0.995,  float(emb_x.loc["t-SNE-2D", "trustworthiness"]),  0.02),
    ("trustworthiness UMAP",  rc["tw_umap"],      0.9884, float(emb_x.loc["UMAP-2D", "trustworthiness"]),   0.02),
]
print(f"{'metrica':24s}{'recomputado':>12s}{'paper/ref':>11s}{'Excel':>9s}{'tol':>7s}")
print("-" * 63)
for nombre, rec, ref, xls, tol in filas:
    print(f"{nombre:24s}{rec:>12.4f}{ref:>11.4f}{xls:>9.4f}{tol:>7.2f}")
    assert abs(rec - xls) <= tol, (nombre, rec, xls, tol)     # el Excel coincide con lo REAJUSTADO desde la base
print("OK: recomputado ~ Excel (t-SNE/UMAP REAJUSTADOS desde digits, tol 0.02; PCA determinista, tol 0.01)")

📖 **Cómo se lee.** Las **cuatro** métricas deterministas del PCA, **recomputadas desde los datos**, coinciden con `S06_resultados.xlsx` dentro de 0.01; y las **dos** estocásticas (trustworthiness de t-SNE/UMAP), **reajustadas desde cero desde `digits.data`**, coinciden dentro de 0.02 (holgura por el no determinismo de openTSNE/UMAP). Como el reajuste es **independiente** del embedding que alimentó el Excel (celda 71), esta verificación **deja de ser tautológica**: prueba que el Excel es **producto de ejecutar el código**, no un registro tecleado. La verificación formal con tolerancias frente al paper vive en el material de referencia de la sesión, que **recomputa desde la base** y cruza recomputado ≈ paper ≈ Excel.

## 6.3 en profundidad — 🧱 Construcción del PCA desde cero (Sección 7 del cuaderno)

Se rearma el PCA **sin `sklearn.PCA`** —solo `numpy`— desde los arrays crudos, para probar que se entiende la mecánica: centrar (o estandarizar), formar la covarianza, resolver los autovalores y quedarse con la razón de varianza. El flujo reproduce el **contrato** (Iris 0.9246 / 0.7296; digits 21) con `assert`.

🔎 **Qué hace este código.** Define `pca_desde_cero` (solo `numpy`) y la aplica a los arrays crudos de Iris (covarianza y correlación) y de `digits`, reproduciendo el contrato (0.9246 / 0.7296 / 21) con `assert`.

In [ ]:
# 🧱 PCA desde cero: SOLO numpy, sin sklearn.PCA, desde los arrays crudos -> reproduce el contrato
def pca_desde_cero(M, estandarizar=False):
    M = np.asarray(M, dtype=float)
    if estandarizar:
        M = (M - M.mean(0)) / M.std(0, ddof=0)     # z-score -> convencion de correlacion
    Mc = M - M.mean(0)                             # centrar (obligatorio)
    C = (Mc.T @ Mc) / (len(Mc) - 1)                # covarianza (de datos crudos o estandarizados)
    ev = np.sort(np.clip(np.linalg.eigvalsh(C), 0, None))[::-1]
    return ev / ev.sum()                           # razon de varianza explicada

evr_cero_cov = pca_desde_cero(X_iris)                      # Iris covarianza
evr_cero_cor = pca_desde_cero(X_iris, estandarizar=True)   # Iris correlacion
n90_cero_dig = int(np.argmax(np.cumsum(pca_desde_cero(digits.data)) >= 0.90) + 1)

# cruzar con el contrato escrito en el Excel
ve_x = pd.read_excel(XLSX, sheet_name="varianza_explicada")
def _xls(ds, conv, comp):
    mm = (ve_x.dataset == ds) & (ve_x.convencion == conv) & (ve_x.componente == comp)
    return float(ve_x[mm]["varianza_explicada"].iloc[0])
xls_dig = int(pd.read_excel(XLSX, sheet_name="reduccion_pca").set_index("metrica")["valor"]["digits_n_comp_90"])

print(f"Iris cov  PC1 desde cero = {evr_cero_cov[0]:.4f} | Excel = {_xls('Iris','covarianza',1):.4f} | target 0.9246")
print(f"Iris corr PC1 desde cero = {evr_cero_cor[0]:.4f} | Excel = {_xls('Iris','correlacion',1):.4f} | target 0.7296")
print(f"digits >=90% desde cero  = {n90_cero_dig:>6d} | Excel = {xls_dig:>6d} | target 21")

assert abs(evr_cero_cov[0] - 0.9246) <= 0.005
assert abs(evr_cero_cor[0] - 0.7296) <= 0.007
assert abs(evr_cero_cov[0] - _xls("Iris", "covarianza", 1)) <= 1e-3
assert abs(evr_cero_cor[0] - _xls("Iris", "correlacion", 1)) <= 1e-3
assert n90_cero_dig == 21
print("OK: el PCA reconstruido con numpy reproduce el contrato (0.9246 / 0.7296 / 21)")

📖 **Cómo se lee.** Con **una función de `numpy`** se reproduce el contrato: covarianza → 0.9246, correlación → 0.7296, digits → 21 componentes para el 90 %. `sklearn.PCA` no hace más que esta eigen-descomposición.

✍️ **Ahora, por cuenta propia.** Extender `pca_desde_cero` para que devuelva también los **autovectores** (las cargas) y reconstruir con ellos el **biplot** de Wine estandarizado del Drill 2, sin usar `sklearn.PCA`.

## 6.6 — ¿Cuándo se puede confiar en una proyección? Supuestos: decisiones y condiciones de validez

> **Fuente canónica:** la guía de supuestos de la sesión. Estas técnicas son **no supervisadas**: no modelan una `Y` con un residuo, así que **no** tienen los supuestos distribucionales del OLS/GLM (nada de normalidad ni homocedasticidad). Lo que sí tienen son **decisiones de preparación** (estandarizar), **supuestos sobre la estructura** (linealidad; varianza = información), **pruebas de adecuación** del Análisis Factorial (KMO, Bartlett) y **reglas de lectura** de los mapas no lineales. Los diagnósticos de esta sección **no escriben en el Excel de contrato**.

| Punto (SUPUESTOS_S06.md) | Cómo se identifica | Decisión / corrección |
|---|---|---|
| **1.1 Estandarización (cov vs. corr)** — central | escalas dispares; 1 componente ~99 % = alarma | estandarizar (correlación) por defecto; documentar |
| **1.3 Linealidad** | PCA-2D ≪ t-SNE/UMAP en el proxy | t-SNE/UMAP si la estructura es no lineal |
| **1.4 Varianza = información** | PC1 de alta varianza sin poder discriminante | PLS si hay `Y`; LDA (S10) para clasificar |
| **1.5 Sensibilidad a atípicos** | un caso solitario «tira» del eje; PCA con y sin el caso | investigar el caso; escalado robusto; robust PCA (avanzado) |
| **2.1 KMO / 2.2 Bartlett** | KMO ≥ 0.60; Bartlett `p<0,05` | doble condición de entrada al Análisis Factorial |
| **2.3 Kaiser + scree** | λ>1; codo; varianza acumulada | cruzar los tres; nunca Kaiser solo |
| **3.1 No deterministas** | el mapa cambia con la semilla | fijar `random_state`; confirmar con varias ejecuciones |
| **3.2 / 3.4 Distancias y ejes** | tamaños/distancias engañosos; ejes arbitrarios | leer solo vecindad; no interpretar ejes ni distancias |
| **3.6 Trustworthiness** | falsos vecinos; 0–1, →1 mejor | reportarla; no extenderla a lo global |

Se ejecutan **seis** diagnósticos: **8.1** la decisión central (covarianza vs. correlación), **8.2** la adecuación del Análisis Factorial (KMO + Bartlett + Kaiser), **8.3** el no determinismo de t-SNE/UMAP, **8.4** la trustworthiness como métrica de fidelidad, **8.5** el supuesto «varianza = información» puesto a prueba en un caso donde **falla**, y **8.6** la **sensibilidad a atípicos** medida con el PCA ajustado **con y sin** el caso influyente.

### Estandarización: covarianza vs. correlación — la decisión central (Parte 1.1) — capítulo 6.6 (subsección 8.1)

🔎 **Qué hace este código.** Compara, para Iris y Wine, el PC1 por covarianza vs. correlación y el nº de componentes ≥90 % en cada convención (diagnóstico; **no** escribe el Excel).

In [ ]:
# Diagnostico 8.1 - la decision estrella: covarianza vs correlacion (NO escribe el Excel)
for nombre, data in [("Iris", X_iris), ("Wine", wine.data)]:
    evr_c = PCA().fit(data).explained_variance_ratio_
    evr_z = PCA().fit(StandardScaler().fit_transform(data)).explained_variance_ratio_
    n90_c = int(np.argmax(np.cumsum(evr_c) >= 0.90) + 1)
    n90_z = int(np.argmax(np.cumsum(evr_z) >= 0.90) + 1)
    print(f"{nombre:5s}  PC1 cov={evr_c[0]*100:5.1f}%  PC1 corr={evr_z[0]*100:5.1f}%  "
          f"| n comp >=90%: crudo={n90_c}  estandarizado={n90_z}")
print("\nRegla: que un solo componente 'explique' ~99% (Wine crudo) es ALARMA de no estandarizar,")
print("no una buena noticia. Estandarizar (correlacion) es la convencion por defecto.")

📖 **Cómo se lee.** La misma nube da lecturas distintas: en **Iris** el PC1 pasa de **92.5 % (covarianza)** a **73.0 % (correlación)**; en **Wine** —variables de escalas dispares— el PC1 crudo es un **99.8 % artificial** (solo refleja `proline`) y hacen falta **8** componentes al estandarizar. ⚠️ **Regla:** si las unidades difieren, se **debe** estandarizar (usar la correlación) y **documentar** la convención. Desarrollo completo en la guía de supuestos de la sesión, Parte 1.1.

### Adecuación del Análisis Factorial: KMO + esfericidad de Bartlett + Kaiser (Partes 2.1, 2.2, 2.3) — capítulo 6.6 (subsección 8.2)

🔎 **Qué hace este código.** Comprueba la adecuación del Análisis Factorial en Iris y Wine con el **KMO** y la **esfericidad de Bartlett**, y cuenta los factores por Kaiser (diagnóstico; **no** escribe el Excel).

In [ ]:
# Diagnostico 8.2 - adecuacion del Analisis Factorial: KMO + Bartlett + Kaiser (NO escribe el Excel)
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
for nombre, data in [("Iris", load_iris().data), ("Wine", load_wine().data)]:
    Xs = StandardScaler().fit_transform(data)
    chi2, p = calculate_bartlett_sphericity(Xs)
    _, kmo = calculate_kmo(Xs)
    eig = np.sort(np.linalg.eigvalsh(np.corrcoef(data, rowvar=False)))[::-1]
    kaiser = int((eig > 1).sum())
    veredicto = "adecuado (>=0.60)" if kmo >= 0.60 else "mediocre (<0.60)"
    factoriz = "rechaza identidad -> factorizable" if p < 0.05 else "no rechaza"
    print(f"{nombre:5s}  KMO={kmo:.3f} [{veredicto}]  |  Bartlett chi2={chi2:.0f} p={p:.1e} ({factoriz})  "
          f"|  Kaiser (lambda>1)={kaiser}")

📖 **Cómo se lee.** El **KMO** mide qué parte de la varianza podría deberse a factores comunes (escala de Kaiser: ≥0.90 excelente, ≥0.80 muy bueno, ≥0.70 aceptable, ≥0.60 mínimo tolerable, <0.50 inaceptable). **Wine** (KMO ≈ 0.78) es **adecuado** para factorizar; **Iris** (KMO ≈ 0.54, solo 4 variables) queda **por debajo del mínimo** —un caso límite útil para ver que la prueba discrimina—. La **esfericidad de Bartlett** rechaza en ambos (`p ≪ 0.05`): existen correlaciones que factorizar; se lee **junto** al KMO porque con `n` grande casi siempre rechaza. El **criterio de Kaiser** (λ>1) sugiere **1** factor en Iris y **3** en Wine. ⚠️ Doble condición de entrada: **KMO ≥ 0.60 y `p<0,05`**. Desarrollo en la guía de supuestos de la sesión, Partes 2.1–2.3.

### t-SNE y UMAP no son deterministas; los ejes y las distancias no se interpretan (Partes 3.1, 3.2, 3.4) — capítulo 6.6 (subsección 8.3)

**❓ Qué se quiere averiguar.** En el mapa anterior hay agrupaciones que quedan lejos entre sí y otras contiguas. ¿Autoriza eso a afirmar que esos grupos «se parecen más» unos a otros?

- **Qué decide:** de la respuesta depende qué frases pueden escribirse en un informe que lleve un t-SNE o un UMAP dentro. «Estos dos segmentos están al doble de distancia que aquellos» es justo la afirmación que se pone a prueba aquí.
- **Antes de mirar el resultado:** si el mismo dato con **otra semilla** devolviera el mismo mapa, las posiciones serían un hecho del dato y la distancia entre agrupaciones se podría leer. Si en cambio la **trustworthiness** apenas se mueve mientras la **disparidad de Procrustes** entre las dos ejecuciones resulta claramente **por encima de 0** —contra el 0,0000 de una semilla consigo misma—, entonces la fidelidad local es real y la geometría del mapa no lo es: los ejes no significan nada, y ni la distancia, ni el tamaño, ni la densidad de una agrupación se interpretan.

🔎 **Qué hace este código.** Ejecuta t-SNE y UMAP **dos veces con semillas distintas** sobre un subconjunto: compara la trustworthiness (estable) y mide con la **disparidad de Procrustes** cuánto difieren los dos mapas de t-SNE (0 = idénticos). Diagnóstico; **no** escribe el Excel.

In [ ]:
# Diagnostico 8.3 - t-SNE/UMAP NO son deterministas: dos semillas -> mapas distintos, fidelidad estable
from scipy.spatial import procrustes
sub = np.random.default_rng(0).choice(len(yf), size=700, replace=False)
Xs2 = Xf[sub]
print(f"{'tecnica':8s}{'semilla':>9s}{'trustworthiness':>17s}")
emb_seed = {}
for seed in (0, 7):
    et = np.asarray(TSNE_open(n_components=2, perplexity=30, initialization="random",
                              random_state=seed, n_jobs=-1).fit(Xs2))
    eu = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=seed).fit_transform(Xs2)
    emb_seed[seed] = et
    print(f"{'t-SNE':8s}{seed:>9d}{trustworthiness(Xs2, et, n_neighbors=5):>17.4f}")
    print(f"{'UMAP':8s}{seed:>9d}{trustworthiness(Xs2, eu, n_neighbors=5):>17.4f}")
# ¿es el mismo mapa? disparidad de Procrustes tras la mejor alineacion posible (0 = identicos)
_, _, disp_dif = procrustes(emb_seed[0], emb_seed[7])   # semillas distintas
_, _, disp_ig  = procrustes(emb_seed[0], emb_seed[0])   # control: misma semilla
print(f"\nProcrustes t-SNE: semillas distintas = {disp_dif:.4f}  vs  misma semilla = {disp_ig:.4f}")
print("-> trustworthiness casi identica (misma fidelidad LOCAL) pero el mapa NO es el mismo:")
print("   cambia con la semilla (rotado/reflejado, grumos recolocados) -> los EJES no significan")
print("   nada y la distancia/tamano de los grumos NO se interpretan.")

📖 **Cómo se lee.** Con **dos semillas distintas** la **trustworthiness** de t-SNE (y de UMAP) es **casi idéntica** —preservan la **misma estructura local**— pero **el mapa no es el mismo**: la **disparidad de Procrustes** entre semillas distintas es **> 0** (frente a 0.0000 con la misma semilla), es decir, el mapa aparece **rotado/reflejado y con las agrupaciones recolocadas**. Por eso **una coordenada absoluta no es interpretable: los ejes no significan nada**. ⚠️ Y **no** se leen del mapa la **distancia** entre agrupaciones, el **tamaño** de una agrupación ni su **densidad** (las colas t de t-SNE inflan las separaciones a propósito). Se fija `random_state` para reproducir y se confirma la estructura con **varias ejecuciones**. Desarrollo en la guía de supuestos de la sesión, Partes 3.1, 3.2 y 3.4.

### Fidelidad de un embedding: trustworthiness (Parte 3.6) — capítulo 6.7 (subsección 8.4)

🔎 **Qué hace este código.** Muestra la tabla de kNN/trustworthiness por representación (desde la hoja `embeddings` del Excel) y resalta el matiz local vs. global. Diagnóstico; **no** escribe el Excel.

In [ ]:
# Diagnostico 8.4 - trustworthiness como metrica de fidelidad LOCAL (NO escribe el Excel)
tabla = pd.read_excel(XLSX, sheet_name="embeddings")
print(tabla.to_string(index=False))
print("\ntrustworthiness en [0,1]; ->1 mejor. Penaliza los 'falsos vecinos' (puntos cercanos en el")
print("mapa que NO lo eran en 64D). t-SNE 0.995 y UMAP 0.988 > PCA-2D 0.830: preservan la vecindad.")
print("MATIZ: una trustworthiness alta prueba fidelidad LOCAL, NO que las distancias globales del")
print("       mapa sean fiables (ver SUPUESTOS_S06.md, Partes 3.2 y 3.6).")

📖 **Cómo se lee.** La **trustworthiness** convierte «el mapa se ve bien» en una **cifra** reproducible: penaliza los **falsos vecinos** (puntos cercanos en 2D que no lo eran en 64D), va de 0 a 1 y cuanto más cerca de 1, más fiel es la **estructura local**. t-SNE (**0.995**) y UMAP (**0.988**) superan al PCA-2D (**0.830**), lo que confirma que recuperan la vecindad que el PCA lineal pierde. ⚠️ **Matiz crítico:** una trustworthiness alta prueba fidelidad **local**, **no** que las distancias **globales** del mapa sean exactas (Partes 3.2 y 3.6 son ciertas a la vez). Desarrollo en la guía de supuestos de la sesión, Parte 3.6.

### «Varianza = información»: cuándo el eje de más varianza NO es el informativo (Parte 1.4) — capítulo 6.6 (subsección 8.5)


🔎 **Qué hace este código.** Construye un caso donde el supuesto **falla a propósito**: dos grupos separados en una dirección de **poca** varianza y solapados en la de **mucha**. Ajusta el PCA, mide qué fracción de varianza se lleva el PC1 y entrena un clasificador **usando solo PC1** y **usando solo PC2** para ver cuál de los dos ejes lleva la información. Después ancla la lección en un dato **real** de la sesión (PCR vs. PLS sobre Communities, hoja `curva_cv_pcr`). Diagnóstico; **no** escribe el Excel.


In [ ]:
# Diagnostico 8.5 - varianza != informacion: el PC1 se lleva la varianza, el PC2 la senal
# (NO escribe el Excel). Caso construido a proposito, semilla fija -> reproducible.
from sklearn.linear_model import LogisticRegression
_rng = np.random.default_rng(RANDOM_STATE)
_n = 300
eje_ruidoso = _rng.normal(0, 10.0, size=2 * _n)                      # MUCHA varianza, cero senal
eje_informativo = np.concatenate([_rng.normal(-1.0, 0.30, _n),       # POCA varianza, toda la senal
                                  _rng.normal(+1.0, 0.30, _n)])
X_demo = np.column_stack([eje_ruidoso, eje_informativo])
y_demo = np.concatenate([np.zeros(_n), np.ones(_n)]).astype(int)

pca_demo = PCA().fit(X_demo)
Z_demo = pca_demo.transform(X_demo)
acc_pc1 = float(cross_val_score(LogisticRegression(), Z_demo[:, [0]], y_demo, cv=5).mean())
acc_pc2 = float(cross_val_score(LogisticRegression(), Z_demo[:, [1]], y_demo, cv=5).mean())
print(f"PC1 se lleva el {pca_demo.explained_variance_ratio_[0]*100:.2f}% de la varianza  ->  "
      f"exactitud clasificando SOLO con PC1 = {acc_pc1:.4f}  (azar = 0.5)")
print(f"PC2 se lleva el {pca_demo.explained_variance_ratio_[1]*100:.2f}% de la varianza  ->  "
      f"exactitud clasificando SOLO con PC2 = {acc_pc2:.4f}")
print("Quien 'reduzca a 1 componente porque explica el 98.8% de la varianza' TIRA justo el eje")
print("que separaba los grupos. El PCA optimiza VARIANZA, no separacion ni prediccion: es")
print("NO SUPERVISADO y nunca mira la etiqueta ni la Y.")

# --- El mismo supuesto, sobre datos REALES de la sesion (hoja `curva_cv_pcr`) ---
_cv = pd.read_excel(XLSX, sheet_name="curva_cv_pcr")
_min_pcr = _cv[(_cv["modelo"] == "PCR") & (_cv["es_minimo"] == 1)].iloc[0]
_min_pls = _cv[(_cv["modelo"] == "PLS") & (_cv["es_minimo"] == 1)].iloc[0]
print(f"\nAncla real (Communities and Crime): la PCR necesita k={int(_min_pcr['n_componentes'])} "
      f"componentes de MAXIMA VARIANZA (MSE de CV {_min_pcr['mse_cv_5fold']:.5f}) para lo que la")
print(f"PLS logra con k={int(_min_pls['n_componentes'])} ({_min_pls['mse_cv_5fold']:.5f}), porque "
      f"la PLS SI mira la Y al construir sus componentes.")
print("CORRECCION: con una Y, usar PLS en vez de PCR; para clasificar con etiquetas, la reduccion")
print("supervisada es el LDA (frontera S10, aqui solo se nombra). Ver SUPUESTOS_S06.md, Parte 1.4.")


📖 **Cómo se lee.** El PC1 se lleva el **98,82 %** de la varianza y, aun así, clasificar con él solo acierta **0,4933** —puro azar—; el PC2, con el **1,18 %** restante, acierta **0,9983**. Toda la información estaba en el eje que un criterio de varianza habría descartado primero. Es el supuesto **1.4** de `SUPUESTOS_S06.md` visto por dentro: *«máxima varianza» y «máxima información» no son sinónimos*. **Cómo se identifica** en la práctica: si al reducir cae de forma marcada el desempeño de un modelo (o la separación de las clases) pese a conservar mucha varianza, la dirección útil quedó fuera. **Cómo se decide:** cuando hay una `Y`, elegir las direcciones mirándola —**PLS**, que en Communities iguala a la PCR con **10** componentes frente a **60**—; para clasificar con etiquetas, la reducción supervisada es el **LDA** (**[Frontera S10]**, aquí solo se nombra). El PCA sigue siendo la herramienta correcta cuando el objetivo es **comprimir o visualizar sin etiqueta**: el error no es usarlo, es olvidar que no mira la respuesta.


### Sensibilidad a valores atípicos: el mismo PCA con y sin un caso influyente (Parte 1.5) — capítulo 6.6 (subsección 8.6)


🔎 **Qué hace este código.** Ejecuta el **análisis de sensibilidad** que `SUPUESTOS_S06.md` (Parte 1.5) pide y que hasta ahora solo estaba descrito: ajusta el PCA de Iris por covarianza **con y sin** un atípico inyectado en una sola observación, y compara tres magnitudes —la varianza explicada del PC1, el **ángulo** entre las dos direcciones de PC1 y la carga de la variable dominante—. Después repite el contraste con **`RobustScaler`** (mediana e IQR) para medir cuánto amortigua la corrección. Diagnóstico; **no** escribe el Excel.


In [ ]:
# Diagnostico 8.6 - sensibilidad a atipicos: el PCA con y sin UN caso influyente
# (NO escribe el Excel). Todo determinista: no hay muestreo aleatorio.
from sklearn.preprocessing import RobustScaler
X_sin = load_iris().data.copy()
X_con = X_sin.copy()
_j = 2                                        # 'petal length (cm)', la variable dominante del PC1
_valor_original = float(X_con[0, _j])
X_con[0, _j] = 30.0                           # un solo dato mal capturado: 30 cm de petalo

def _perfil(X):
    p = PCA().fit(X)
    return p.explained_variance_ratio_[:2], p.components_[0]

(ve_sin, v_sin), (ve_con, v_con) = _perfil(X_sin), _perfil(X_con)
ang = float(np.degrees(np.arccos(min(1.0, abs(float(v_sin @ v_con))))))
print(f"Un unico dato alterado (obs. 0: largo de petalo {_valor_original:.1f} cm -> 30.0 cm) sobre 150:")
print(f"  PC1 varianza explicada : {ve_sin[0]:.4f}  ->  {ve_con[0]:.4f}   (PC2 {ve_sin[1]:.4f} -> {ve_con[1]:.4f})")
print(f"  carga v del petalo     : {v_sin[_j]:+.4f}  ->  {v_con[_j]:+.4f}   (el eje se gira HACIA el atipico)")
print(f"  angulo entre los dos PC1: {ang:.2f} grados  -> NO es el mismo eje: el biplot, el nombre del")
print("    componente y la k de la PCR cambian por UN dato (0.67% de la muestra).")

# --- Correccion: escalado ROBUSTO (mediana e IQR en vez de media y desviacion tipica) ---
pr_sin = PCA().fit(RobustScaler().fit_transform(X_sin))
pr_con = PCA().fit(RobustScaler().fit_transform(X_con))
ang_rob = float(np.degrees(np.arccos(min(1.0, abs(float(pr_sin.components_[0] @ pr_con.components_[0]))))))
print(f"\nCon RobustScaler (mediana / IQR): el giro del PC1 baja de {ang:.2f} a {ang_rob:.2f} grados")
print("-> amortigua, NO inmuniza. La regla de SUPUESTOS_S06.md (Parte 1.5): investigar primero el")
print("   caso (?error de captura?), reportar el PCA CON y SIN el, y escalar de forma robusta; el")
print("   robust PCA (MCD/ROBPCA) solo se nombra como la correccion avanzada.")


📖 **Cómo se lee.** Un **solo** dato mal capturado —el 0,67 % de la muestra— mueve el PC1 de Iris de **0,9246** a **0,8811** de varianza explicada, sube la carga del largo de pétalo de **0,8567** a **0,9740** y **gira el eje 17,96°**: no es el mismo componente, aunque se siga llamando «PC1». Ahí está la razón de fondo: el PCA se apoya en varianzas y covarianzas —desviaciones **al cuadrado**—, así que un punto lejano pesa desproporcionadamente y **arrastra el eje hacia sí**. Las consecuencias son las tres que la sesión califica: el **biplot** apunta a otro sitio, el **nombre** del componente cambia y la **k** que se elige por varianza acumulada se desplaza. **Cómo se identifica:** un punto solitario que «tira» de un eje en el biplot, z-scores extremos por variable y este mismo **análisis con y sin el caso** (enlaza con los influyentes de S04: Cook y *leverage*, aquí aplicados a la estimación de la covarianza). **Cómo se decide:** investigar primero si es un error de datos; reportar el PCA **con y sin**; usar **`RobustScaler`** (mediana e IQR), que aquí reduce el giro de 17,96° a **10,19°** —lo amortigua, no lo elimina—; el **robust PCA** (MCD/ROBPCA) queda **[Avanzado]**, solo nombrado. Detalle en la guía de supuestos de la sesión, Parte 1.5.


## Práctica — Drills (ejercicios) (Sección 9 del cuaderno)

Resolver en este mismo notebook, reutilizando los objetos ya cargados. Se entregan los **enunciados**; la solución es parte del trabajo del estudiante. Ver también `evaluacion/drills.docx`.

**Drill 1 — Elegir cuántos componentes retener.**
Sobre `load_wine()` **estandarizado**, graficar la **varianza acumulada** por número de componentes y trazar la línea del **90 %**. Combinar el **scree plot** (¿hay codo?), el **criterio de Kaiser** (eigenvalor > 1) y el umbral del 90 %, y justificar cuántos componentes retener cuando los tres criterios **no coinciden**.

**Drill 2 — Interpretar de negocio los dos primeros componentes de un biplot.**
Construir el **biplot** PC1-PC2 de `load_wine()` estandarizado (observaciones + flechas de las cargas). Nombrar PC1 y PC2 según las variables que más cargan (p. ej. "cuerpo/riqueza fenólica" y "perfil de color"), y explicar cómo se ordenan los tres cultivares sobre esos ejes. No interpretar el **signo** de una carga como bueno/malo.

**Drill 3 — Comparar t-SNE vs. UMAP variando hiperparámetros y discutir estabilidad.**
Sobre `load_digits()`, generar mapas 2D con **t-SNE** para `perplexity ∈ {5, 30, 50}` y con **UMAP** para `n_neighbors ∈ {5, 15, 50}` (y probar dos valores de `min_dist`). Discutir qué agrupaciones se mantienen **estables** a través de los hiperparámetros y cuáles cambian, y argumentar por qué **no** se debe leer un único mapa como conclusión.

---

## 6.9 — ¿Qué no se puede afirmar, y qué sigue en S07? Cierre (Sección 10 del cuaderno)

### Entregable evaluable
Aplicar **PCA** (elegir el nº de componentes por varianza explicada), **interpretar** los dos primeros componentes en un biplot, **comparar** una visualización t-SNE vs. UMAP, y usar **PCR** para combatir la multicolinealidad de un dataset previo, cerrando con una **recomendación** (modelado con PCA/PCR vs. visualización con t-SNE/UMAP, según el objetivo). Calificación vigesimal (0–20); enunciado y rúbrica en `evaluacion/entregable.docx`.

### Control corto
La sesión cierra con un control corto (banco de ítems `[S06]`) sobre: varianza explicada y elección del nº de componentes, convención covarianza vs. correlación, diferencia PCA/Análisis Factorial, PCR contra la multicolinealidad, y los **límites** de t-SNE/UMAP (qué NO se debe leer de un mapa).

### Proyecto integrador
Esta sesión alimenta la fase de **Preparación/Modelado**: en el dataset del proyecto se usará el PCA/PCR para reducir dimensiones y quitar multicolinealidad, y t-SNE/UMAP para **explorar visualmente** la estructura antes de modelar —siempre distinguiendo visualización de modelado—.

### Materiales de apoyo de la sesión
- Guía del laboratorio: `laboratorio/GUIA_LABORATORIO_S06.docx`
- Guía para interpretar componentes y factores: `plantillas/guia_componentes_factores.docx`
- Plantilla de comparación PCA / t-SNE / UMAP: `plantillas/comparacion_pca_tsne_umap.docx`
- Mapa de pasos del cuaderno (celda↔slide): el cuaderno de la sesión
- Supuestos y decisiones (fuente canónica): la guía de supuestos de la sesión
- Drills y entregable: `evaluacion/drills.docx`, `evaluacion/entregable.docx`

### Para seguir explorando (actualidad — ver las fuentes de actualidad de la sesión)
- **UMAP para monitorear embeddings de LLM/RAG en producción** — *Embedding Visualization with UMAP* (Fiddler AI, 2026): proyectar embeddings de 384-4096 dimensiones a 2D para detectar *drift* y *outliers*.
- **La visualización de embeddings como parte del stack de IA de 2026** — *What Is Embedding Visualization?* (FutureAGI, 2026): visualizar para generar hipótesis y luego confirmar con métricas.
- **PCA/UMAP para abaratar el almacenamiento de embeddings en RAG** — arXiv 2505.00105 (2025): PCA moderada + cuantización logra 8× de compresión con mínima pérdida.
- **Cuándo NO confiar en un mapa 2D** — *Why Can't I See My Clusters?* (arXiv 2509.04222, 2025): métricas de precisión/recall para validar la reducción de dimensiones.
- **Estado del arte comparado de UMAP** — arXiv 2603.02275 (2026): UMAP vs. PCA/Kernel PCA/t-SNE; "UMAP no siempre es mejor".

> **Alcance.** La reducción de dimensiones de esta sesión es para **modelar** (PCA/PCR) o **visualizar** (t-SNE/UMAP). El **clustering** (jerárquico, DBSCAN), la **detección de anomalías** y K-Means pertenecen a la **Sesión 7**: aquí t-SNE/UMAP solo dibujan, no agrupan. Las sesiones S07–S14 (clustering, asociación, GLM, series, causalidad, supervivencia, recomendación) solo se nombran.

---

*Cuaderno de la Sesión 6, Herramientas para la Ciencia de Datos, UPC. Toda cifra proviene de `el material de referencia de la sesión` (fuentes verificadas 18/07/2026) y de los papers declarado para la sesión.*